# 03 · Downscaling Climático (CMIP6 → QDM) y ET futura

Este notebook realiza el **downscaling estadístico** de las variables climáticas futuras usando **Quantile Delta Mapping (QDM)** mensual.

**Modelos CMIP6 utilizados:** `MPI-ESM1-2-LR`, `EC-Earth3-Veg-LR`, `EC-Earth3`  
**Escenarios:** `ssp245`, `ssp585`  
**Variables:** precipitación (PR), temperatura máxima (tasmax) y mínima (tasmin).

El flujo es:

1. Construcción de **tablas QDM mensuales** (estadísticas observadas vs. modeladas).
2. **Descarga RAW** de CMIP6 desde GEE.
3. **Corrección espacial QDM** masiva para precipitación (umbrales 0.1 mm y 0.5 mm) y temperatura.
4. **Detección y reparación de faltantes**.
5. Cálculo de **ET Hargreaves** a partir de la temperatura corregida.


## 3.1 · Tablas QDM mensuales — Precipitación


In [ ]:
import ee
import pandas as pd
import numpy as np
from pathlib import Path
import json

# =====================================================
# 1. MONTAR GOOGLE DRIVE Y CONFIGURAR RUTAS
# =====================================================
# En Google Colab, primero montamos Drive para poder guardar archivos
#from google.colab import drive
#drive.mount('/content/drive')

# Ruta base dentro de tu Drive en la carpeta "Project001"
drive_base = Path("/content/drive/MyDrive/Project001")
drive_base.mkdir(parents=True, exist_ok=True)

# Directorio de salida dentro del proyecto
output_dir = drive_base / "salidas_qdm_mensual"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Archivos se guardarán en: {output_dir}")

# =====================================================
# 2. INICIALIZAR EARTH ENGINE
# =====================================================
# En Colab se recomienda autenticación explícita la primera vez
try:
    ee.Initialize(project='computer-492420')  # <-- pon tu project ID aquí
except Exception:
    ee.Authenticate()
    ee.Initialize(project='computer-492420')

# =====================================================
# 3. PARÁMETROS GENERALES
# =====================================================
roi_fc   = ee.FeatureCollection("projects/mapas2025-473512/assets/AreaAporte")
roi_geom = roi_fc.geometry()

start_date = "1985-01-01"
end_date   = "2014-12-31"

modelos = [
    "MPI-ESM1-2-LR",
    "EC-Earth3-Veg-LR",
    "EC-Earth3"
]

scale_chirps = 5000
scale_cmip6  = 5000
n_quantiles  = 101

# ─── NUEVO: dos umbrales de día seco que se procesarán en paralelo ───
UMBRALES_SECOS = [0.1, 0.5]   # mm/día

# Mínimo de datos válidos por mes para construir la tabla QDM
min_muestras_mes = 20

# =====================================================
# 4. FUNCIONES AUXILIARES
# =====================================================

def extraer_serie_temporal(collection, var_name, escala, start_date, end_date, es_cmip=False):
    """
    Extrae serie temporal diaria promediada sobre el ROI.
    Si es_cmip=True convierte pr de kg m-2 s-1 → mm/día (× 86400).
    """
    def reduce_region(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi_geom,
            scale=escala,
            maxPixels=1e9
        ).get(var_name)
        if es_cmip:
            val = ee.Number(val).multiply(86400.0)
        return img.set("valor", val).set("date", img.date().format("YYYY-MM-dd"))

    col      = collection.filterDate(start_date, end_date).map(reduce_region)
    fechas   = col.aggregate_array("date").getInfo()
    valores  = col.aggregate_array("valor").getInfo()

    df = pd.DataFrame({
        "date" : pd.to_datetime(fechas, errors="coerce"),
        "value": pd.to_numeric(valores, errors="coerce")
    })
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df


def limpiar_serie(arr, umbral_seco=0.1):
    """
    Limpia una serie de precipitación:
      - convierte a float
      - elimina no-finitos
      - fuerza no negativos (clip)
      - aplica umbral de día seco (valores < umbral → 0)
    """
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    arr = np.clip(arr, 0.0, None)
    arr[arr < umbral_seco] = 0.0
    return arr


def asegurar_monotonia(arr):
    """Fuerza un arreglo a ser no decreciente (requerido por una CDF válida)."""
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0:
        return arr
    return np.maximum.accumulate(arr)


def validar_probabilidades(probs):
    """Verifica que el vector de probabilidades sea válido: [0,1] y no decreciente."""
    probs = np.asarray(probs, dtype=float)
    if probs.size == 0:
        return False
    if probs[0] != 0.0 or probs[-1] != 1.0:
        return False
    if np.any(np.diff(probs) < 0):
        return False
    return True


def construir_tabla_qdm_mensual(df, n_quantiles=101, umbral_seco=0.1, min_muestras_mes=20):
    """
    Construye la tabla de cuantiles mensuales QDM para un par
    (observado CHIRPS, modelo CMIP6) y un umbral de día seco dado.

    Retorna un DataFrame con 12 filas (una por mes) que contiene:
      - probs        : vector de probabilidades [0,1] serializado en JSON
      - q_obs_hist   : cuantiles observados históricos (JSON)
      - q_mod_hist   : cuantiles del modelo histórico (JSON)
      - estadísticas de diagnóstico
    """
    probs = np.linspace(0.0, 1.0, n_quantiles)
    rows  = []

    for mes in range(1, 13):
        sub = df[df["mes"] == mes].copy()

        # Limpiar ambas series con el umbral indicado
        obs = limpiar_serie(sub["pr_obs"].to_numpy(dtype=float), umbral_seco=umbral_seco)
        mod = limpiar_serie(sub["pr_mod"].to_numpy(dtype=float), umbral_seco=umbral_seco)

        # Si no hay suficientes datos, marcar como insuficiente
        if len(obs) < min_muestras_mes or len(mod) < min_muestras_mes:
            rows.append({
                "mes"          : mes,
                "probs"        : json.dumps([]),
                "q_obs_hist"   : json.dumps([]),
                "q_mod_hist"   : json.dumps([]),
                "n_obs"        : int(len(obs)),
                "n_mod"        : int(len(mod)),
                "min_obs"      : np.nan,
                "max_obs"      : np.nan,
                "min_mod"      : np.nan,
                "max_mod"      : np.nan,
                "pct_ceros_obs": np.nan,
                "pct_ceros_mod": np.nan,
                "estado"       : "insuficiente_muestra"
            })
            continue

        # Calcular cuantiles y garantizar validez estadística
        q_obs = asegurar_monotonia(np.clip(np.quantile(obs, probs), 0.0, None))
        q_mod = asegurar_monotonia(np.clip(np.quantile(mod, probs), 0.0, None))

        rows.append({
            "mes"          : mes,
            "probs"        : json.dumps(probs.tolist(), ensure_ascii=False),
            "q_obs_hist"   : json.dumps(q_obs.tolist(), ensure_ascii=False),
            "q_mod_hist"   : json.dumps(q_mod.tolist(), ensure_ascii=False),
            "n_obs"        : int(len(obs)),
            "n_mod"        : int(len(mod)),
            "min_obs"      : float(np.min(obs)),
            "max_obs"      : float(np.max(obs)),
            "min_mod"      : float(np.min(mod)),
            "max_mod"      : float(np.max(mod)),
            "pct_ceros_obs": float((obs == 0).mean() * 100.0),
            "pct_ceros_mod": float((mod == 0).mean() * 100.0),
            "estado"       : "ok"
        })

    return pd.DataFrame(rows)


def validar_tabla_qdm(qdm_df, n_quantiles=101):
    """
    Audita la tabla QDM construida y emite un reporte de validación
    indicando para cada mes si es válida y, si no, el motivo.
    """
    resultados = []

    for _, row in qdm_df.iterrows():
        mes    = int(row["mes"])
        estado = row["estado"]

        if estado != "ok":
            resultados.append({"mes": mes, "valido": False, "motivo": estado})
            continue

        probs = np.array(json.loads(row["probs"]),       dtype=float)
        q_obs = np.array(json.loads(row["q_obs_hist"]), dtype=float)
        q_mod = np.array(json.loads(row["q_mod_hist"]), dtype=float)

        valido, motivos = True, []

        if len(probs) != n_quantiles: valido = False; motivos.append("len_probs")
        if len(q_obs)  != n_quantiles: valido = False; motivos.append("len_q_obs")
        if len(q_mod)  != n_quantiles: valido = False; motivos.append("len_q_mod")
        if not validar_probabilidades(probs):
            valido = False; motivos.append("probs_no_validas")
        if np.any(np.diff(q_obs) < -1e-12): valido = False; motivos.append("q_obs_no_monotona")
        if np.any(np.diff(q_mod) < -1e-12): valido = False; motivos.append("q_mod_no_monotona")
        if np.any(q_obs < 0): valido = False; motivos.append("q_obs_negativo")
        if np.any(q_mod < 0): valido = False; motivos.append("q_mod_negativo")

        resultados.append({
            "mes"   : mes,
            "valido": valido,
            "motivo": ";".join(motivos) if motivos else "ok"
        })

    return pd.DataFrame(resultados)


def guardar_metadata_modelo(modelo, df_hist, out_dir, n_quantiles, umbral_seco, min_muestras_mes):
    """
    Guarda un CSV con el 'pasaporte' del procesamiento:
    fuentes, período, parámetros clave y notas de trazabilidad.
    """
    meta = pd.DataFrame([{
        "modelo"             : modelo,
        "metodo"             : "QDM_mensual",
        "n_quantiles"        : n_quantiles,
        "fecha_inicio"       : df_hist["date"].min().strftime("%Y-%m-%d"),
        "fecha_fin"          : df_hist["date"].max().strftime("%Y-%m-%d"),
        "n_registros"        : int(len(df_hist)),
        "observado"          : "CHIRPS",
        "modelo_fuente"      : "NASA/GDDP-CMIP6",
        "variable_obs"       : "precipitation",
        "variable_modelo"    : "pr",
        "unidad_final"       : "mm/dia",
        "umbral_seco_mm_dia" : umbral_seco,
        "min_muestras_mes"   : min_muestras_mes,
        "nota"               : (
            "Archivo preparado para corrección futura con QDM mensual. "
            "CSV guardado sin BOM (utf-8). "
            f"Umbral día seco usado: {umbral_seco} mm/día."
        )
    }])
    nombre   = modelo.replace("-", "_")
    tag_u    = str(umbral_seco).replace(".", "p")   # ej. "0.1" → "0p1"
    out_meta = out_dir / f"METADATA_{nombre}_u{tag_u}.csv"
    meta.to_csv(out_meta, index=False, encoding="utf-8")
    print(f"  Guardado: {out_meta.name}")


def resumen_serie(df, col):
    """Estadísticas básicas de una columna numérica."""
    x = pd.to_numeric(df[col], errors="coerce")
    return {
        "n"    : int(x.notna().sum()),
        "min"  : float(x.min()),
        "max"  : float(x.max()),
        "media": float(x.mean()),
        "p95"  : float(x.quantile(0.95))
    }


# =====================================================
# 5. EXTRAER OBSERVADO CHIRPS (solo una vez, común a todos)
# =====================================================
print("\n[1/3] Extrayendo CHIRPS histórico (1985–2014)...")
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")

df_obs = extraer_serie_temporal(
    chirps, "precipitation", scale_chirps, start_date, end_date, es_cmip=False
).rename(columns={"value": "pr_obs"})

df_obs["pr_obs"] = pd.to_numeric(df_obs["pr_obs"], errors="coerce").clip(lower=0.0)

# Guardar observado base (sin umbral aplicado, para trazabilidad)
out_obs = output_dir / "OBS_CHIRPS_historico.csv"
df_obs.to_csv(out_obs, index=False, encoding="utf-8")
print(f"  Guardado: {out_obs.name}")

# =====================================================
# 6. BASE CMIP6 (filtrado temporal previo al bucle)
# =====================================================
cmip6_base = ee.ImageCollection("NASA/GDDP-CMIP6").filterDate(start_date, end_date)

# =====================================================
# 7. BUCLE PRINCIPAL: modelo × umbral
# =====================================================
print(f"\n[2/3] Procesando {len(modelos)} modelos × {len(UMBRALES_SECOS)} umbrales...\n")

for modelo in modelos:
    print(f"══ Modelo: {modelo} ══")
    nombre = modelo.replace("-", "_")

    # ── 7a. Extraer serie CMIP6 del modelo (solo 1 vez por modelo) ──
    cmip6_mod = (
        cmip6_base
        .filter(ee.Filter.eq("model",    modelo))
        .filter(ee.Filter.eq("scenario", "historical"))
        .select("pr")
    )

    df_mod = extraer_serie_temporal(
        cmip6_mod, "pr", scale_cmip6, start_date, end_date, es_cmip=True
    ).rename(columns={"value": "pr_mod"})

    df_mod["pr_mod"] = pd.to_numeric(df_mod["pr_mod"], errors="coerce").clip(lower=0.0)

    # Guardar serie histórica cruda del modelo
    out_mod = output_dir / f"MOD_{nombre}_historico.csv"
    df_mod.to_csv(out_mod, index=False, encoding="utf-8")
    print(f"  Guardado serie modelo: {out_mod.name}")

    # ── 7b. Emparejar observado + modelo por fecha (inner join) ──
    df_base = pd.merge(df_obs, df_mod, on="date", how="inner").dropna().copy()
    df_base["mes"]  = df_base["date"].dt.month.astype(int)
    df_base["anio"] = df_base["date"].dt.year.astype(int)
    df_base["dia"]  = df_base["date"].dt.day.astype(int)

    # Guardar emparejado base (sin umbral aplicado)
    out_hist_base = output_dir / f"HIST_{nombre}_emparejado_base.csv"
    df_base.to_csv(out_hist_base, index=False, encoding="utf-8")
    print(f"  Guardado histórico emparejado base: {out_hist_base.name}")

    # ── 7c. Iterar sobre cada umbral de día seco ──
    for umbral_seco in UMBRALES_SECOS:
        tag_u = str(umbral_seco).replace(".", "p")   # "0.1"→"0p1", "0.5"→"0p5"
        print(f"\n  → Umbral día seco: {umbral_seco} mm/día  (tag={tag_u})")

        # Aplicar umbral al histórico emparejado (copia para no contaminar la base)
        df_hist = df_base.copy()
        df_hist["pr_obs"] = df_hist["pr_obs"].where(df_hist["pr_obs"] >= umbral_seco, 0.0)
        df_hist["pr_mod"] = df_hist["pr_mod"].where(df_hist["pr_mod"] >= umbral_seco, 0.0)

        # Guardar histórico con umbral aplicado
        out_hist = output_dir / f"HIST_{nombre}_u{tag_u}_emparejado.csv"
        df_hist.to_csv(out_hist, index=False, encoding="utf-8")
        print(f"    Guardado: {out_hist.name}")

        # Construir tabla QDM mensual para este umbral
        qdm_df = construir_tabla_qdm_mensual(
            df_hist,
            n_quantiles     = n_quantiles,
            umbral_seco     = umbral_seco,
            min_muestras_mes= min_muestras_mes
        )

        # ── Guardar tabla QDM ──
        out_qdm = output_dir / f"QDM_{nombre}_u{tag_u}_monthly.csv"
        qdm_df.to_csv(out_qdm, index=False, encoding="utf-8")
        print(f"    Guardado: {out_qdm.name}")

        # Validar tabla QDM
        valid_df = validar_tabla_qdm(qdm_df, n_quantiles=n_quantiles)
        out_valid = output_dir / f"VALIDACION_QDM_{nombre}_u{tag_u}.csv"
        valid_df.to_csv(out_valid, index=False, encoding="utf-8")
        print(f"    Guardado: {out_valid.name}")

        # Resumen estadístico en JSON
        resumen = {
            "modelo"           : modelo,
            "umbral_seco_mm"   : umbral_seco,
            "historico_obs"    : resumen_serie(df_hist, "pr_obs"),
            "historico_mod"    : resumen_serie(df_hist, "pr_mod"),
            "meses_validos"    : int(valid_df["valido"].sum()),
            "meses_invalidos"  : int((~valid_df["valido"]).sum())
        }
        out_resumen = output_dir / f"RESUMEN_{nombre}_u{tag_u}.json"
        with open(out_resumen, "w", encoding="utf-8") as f:
            json.dump(resumen, f, ensure_ascii=False, indent=2)
        print(f"    Guardado: {out_resumen.name}")

        # Metadata de trazabilidad
        guardar_metadata_modelo(
            modelo          = modelo,
            df_hist         = df_hist,
            out_dir         = output_dir,
            n_quantiles     = n_quantiles,
            umbral_seco     = umbral_seco,
            min_muestras_mes= min_muestras_mes
        )

# =====================================================
# 8. RESUMEN FINAL
# =====================================================
print("\n[3/3] ¡Proceso completado!")
print(f"  Todos los archivos están en: {output_dir}")
print(f"  Modelos procesados   : {len(modelos)}")
print(f"  Umbrales procesados  : {UMBRALES_SECOS}")
print(f"  Archivos QDM por mod : {len(UMBRALES_SECOS)} (uno por umbral)")
print(f"  Total archivos QDM   : {len(modelos) * len(UMBRALES_SECOS)}")


In [ ]:
import ee
import ast
import numpy as np

# =====================================================
# 1. INICIALIZAR EARTH ENGINE
# =====================================================
# Actualizado al nuevo ID de proyecto: computer-492420
PROJECT_ID = 'computer-492420'

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# =====================================================
# 2. ASSETS A REVISAR (RUTAS ACTUALIZADAS)
# =====================================================
qdm_assets = {
    "MPI-ESM1-2-LR": {
        "u0p1": f"projects/{PROJECT_ID}/assets/QDM_MPI_ESM1_2_LR_u0p1_monthly",
        "u0p5": f"projects/{PROJECT_ID}/assets/QDM_MPI_ESM1_2_LR_u0p5_monthly",
    },
    "EC-Earth3-Veg-LR": {
        "u0p1": f"projects/{PROJECT_ID}/assets/QDM_EC_Earth3_Veg_LR_u0p1_monthly",
        "u0p5": f"projects/{PROJECT_ID}/assets/QDM_EC_Earth3_Veg_LR_u0p5_monthly",
    },
    "EC-Earth3": {
        "u0p1": f"projects/{PROJECT_ID}/assets/QDM_EC_Earth3_u0p1_monthly",
        "u0p5": f"projects/{PROJECT_ID}/assets/QDM_EC_Earth3_u0p5_monthly",
    },
}

UMBRALES_SECOS = {"u0p1": 0.1, "u0p5": 0.5}   # mm/día

campos_esperados = [
    "mes", "probs", "q_obs_hist", "q_mod_hist",
    "n_obs", "n_mod", "min_obs", "max_obs", "min_mod", "max_mod"
]

# =====================================================
# 3. FUNCIONES AUXILIARES
# =====================================================

def parsear_lista(x):
    """Convierte un campo a lista Python, aceptando list, str o None."""
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []


def buscar_campo_mes(props):
    """Busca el campo 'mes' tolerando BOM o espacios."""
    for k, v in props.items():
        nombre_limpio = str(k).replace("\ufeff", "").strip().lower()
        if nombre_limpio == "mes":
            return k, v
    return None, None


def validar_cuantiles(arr, nombre, umbral_seco):
    """Validaciones estadísticas sobre un vector de cuantiles."""
    advertencias = []
    a = np.asarray(arr, dtype=float)

    if np.any(a < 0):
        advertencias.append(f"{nombre}: tiene valores NEGATIVOS")
    if np.any(np.diff(a) < -1e-12):
        advertencias.append(f"{nombre}: NO es monótono no decreciente")
    if a[0] < 0:
        advertencias.append(f"{nombre}: cuantil Q0 negativo")

    return advertencias


def inspeccionar_asset(asset_id, modelo, tag_umbral, umbral_mm):
    """Inspecciona un FeatureCollection QDM en GEE."""
    sep = "=" * 80
    print(f"\n{sep}")
    print(f"MODELO  : {modelo}")
    print(f"UMBRAL  : {umbral_mm} mm/día  ({tag_umbral})")
    print(f"ASSET   : {asset_id}")
    print(sep)

    try:
        fc       = ee.FeatureCollection(asset_id)
        info     = fc.getInfo()
        features = info.get("features", [])

        print(f"Número de features: {len(features)}")

        if len(features) == 0:
            print("[AVISO] El asset no tiene features.")
            return

        props0 = features[0].get("properties", {})
        nombre_mes_real, _ = buscar_campo_mes(props0)

        # Chequeo feature a feature
        print("\nChequeo por cada feature (mes):")
        print(f"  {'Feat':>4}  {'mes':>4}  {'probs':>8}  {'q_obs':>8}  "
              f"{'q_mod':>8}  {'n_obs':>6}  {'n_mod':>6}  Advertencias")
        print("  " + "-" * 100)

        meses_detectados = []
        todo_ok          = True
        todas_advertencias = []

        for i, feat in enumerate(features, start=1):
            props = feat.get("properties", {})
            _, mes_val = buscar_campo_mes(props)

            probs = parsear_lista(props.get("probs"))
            q_obs = parsear_lista(props.get("q_obs_hist"))
            q_mod = parsear_lista(props.get("q_mod_hist"))
            n_obs = props.get("n_obs", "?")
            n_mod = props.get("n_mod", "?")

            meses_detectados.append(mes_val)

            # Validaciones
            adv = []
            if len(probs) != 101: adv.append("probs: len != 101")
            if len(q_obs) != 101: adv.append("q_obs: len != 101")
            if len(q_mod) != 101: adv.append("q_mod: len != 101")

            adv += validar_cuantiles(q_obs, "q_obs", umbral_mm)
            adv += validar_cuantiles(q_mod, "q_mod", umbral_mm)

            if adv: todo_ok = False
            adv_txt = " | ".join(adv) if adv else "—"
            todas_advertencias += adv

            print(
                f"  {i:>4}  {str(mes_val):>4}  "
                f"{len(probs):>8}  {len(q_obs):>8}  {len(q_mod):>8}  "
                f"{str(n_obs):>6}  {str(n_mod):>6}  "
                f"{adv_txt}"
            )

        print(f"\nRESUMEN:")
        secuencia_ok = (len([m for m in meses_detectados if m is not None]) == 12)
        print(f"  Secuencia meses  : {'OK' if secuencia_ok else 'REVISAR'}")
        print(f"  Datos/Stats      : {'OK' if todo_ok else 'REVISAR'}")

    except Exception as e:
        print(f"[ERROR] No se pudo leer el asset: {e}")


# =====================================================
# 4. EJECUCIÓN
# =====================================================
print(f"INSPECCIÓN QDM — PROYECTO: {PROJECT_ID}")

for modelo, assets_por_umbral in qdm_assets.items():
    for tag_umbral, asset_id in assets_por_umbral.items():
        umbral_mm = UMBRALES_SECOS[tag_umbral]
        inspeccionar_asset(asset_id, modelo, tag_umbral, umbral_mm)

In [ ]:
import ee
import ast
import numpy as np
import pandas as pd
from pathlib import Path



# Ruta base del proyecto en Drive (ajusta si tu carpeta tiene otro nombre)
drive_base = Path("/content/drive/MyDrive/Project001/salidas_qdm_mensual")

# =====================================================
# 2. INICIALIZAR EARTH ENGINE (solo para referencia de ROI si se necesita)
#    Si la inspección es 100% local desde Drive, puedes comentar este bloque.
# =====================================================

# =====================================================
# 3. CONFIGURACIÓN DE MODELOS Y UMBRALES
# =====================================================
modelos = [
    "MPI-ESM1-2-LR",
    "EC-Earth3-Veg-LR",
    "EC-Earth3",
]

UMBRALES = {
    "u0p1": 0.1,   # mm/día
    "u0p5": 0.5,   # mm/día
}

n_quantiles_esperado = 101
campos_esperados = [
    "mes", "probs", "q_obs_hist", "q_mod_hist",
    "n_obs", "n_mod", "min_obs", "max_obs", "min_mod", "max_mod",
    "pct_ceros_obs", "pct_ceros_mod", "estado"
]

# =====================================================
# 4. FUNCIONES AUXILIARES
# =====================================================

def parsear_lista(x):
    """Convierte un campo a lista Python: acepta list, str JSON o None."""
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []


def validar_cuantiles(arr, nombre):
    """
    Verifica que un vector de cuantiles sea:
      - no negativo
      - monótono no decreciente
    Retorna lista de advertencias (vacía si todo OK).
    """
    adv = []
    a   = np.asarray(arr, dtype=float)
    if np.any(a < 0):
        adv.append(f"{nombre}: valores NEGATIVOS")
    if np.any(np.diff(a) < -1e-12):
        adv.append(f"{nombre}: NO monótono no decreciente")
    return adv


def cargar_csv_qdm(ruta_csv):
    """
    Carga un CSV QDM desde Drive y retorna un DataFrame.
    Maneja el BOM en el nombre de columnas si existe.
    """
    df = pd.read_csv(ruta_csv, encoding="utf-8")
    # Limpiar BOM en nombres de columnas
    df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]
    return df


def inspeccionar_csv_qdm(ruta_csv, modelo, tag_umbral, umbral_mm):
    """
    Inspecciona un archivo QDM CSV cargado desde Drive.
    Emite diagnóstico completo: campos, secuencia de meses,
    longitud de vectores cuantiles y validaciones estadísticas.
    """
    sep = "=" * 80
    print(f"\n{sep}")
    print(f"MODELO  : {modelo}")
    print(f"UMBRAL  : {umbral_mm} mm/día  ({tag_umbral})")
    print(f"ARCHIVO : {ruta_csv.name}")
    print(sep)

    # ── Verificar existencia del archivo ──
    if not ruta_csv.exists():
        print(f"[ERROR] Archivo no encontrado: {ruta_csv}")
        return

    df = cargar_csv_qdm(ruta_csv)
    print(f"Filas cargadas: {len(df)}")

    # ── Verificar campos esperados ──
    print("\nRevisión de campos esperados:")
    for campo in campos_esperados:
        existe = campo in df.columns
        print(f"  - {campo:<18}: {'OK   ' if existe else 'FALTA'}")

    if "mes" not in df.columns:
        print("\n[ERROR CRÍTICO] No existe columna 'mes'. No se puede continuar.")
        return

    # ── Tabla de diagnóstico feature a feature ──
    print("\nChequeo por cada fila (mes):")
    print(f"  {'mes':>4}  {'estado':<22}  {'probs':>8}  {'q_obs':>8}  "
          f"{'q_mod':>8}  {'n_obs':>6}  {'n_mod':>6}  "
          f"{'pct0_obs':>9}  {'pct0_mod':>9}  Advertencias")
    print("  " + "-" * 120)

    meses_detectados   = []
    todo_ok            = True
    todas_advertencias = []

    for _, row in df.iterrows():
        mes_val  = row.get("mes",    None)
        estado   = row.get("estado", "?")
        n_obs    = row.get("n_obs",  "?")
        n_mod    = row.get("n_mod",  "?")
        p0_obs   = row.get("pct_ceros_obs", "?")
        p0_mod   = row.get("pct_ceros_mod", "?")

        probs = parsear_lista(row.get("probs",       None))
        q_obs = parsear_lista(row.get("q_obs_hist",  None))
        q_mod = parsear_lista(row.get("q_mod_hist",  None))

        meses_detectados.append(mes_val)

        ok_probs = len(probs) == n_quantiles_esperado
        ok_qobs  = len(q_obs) == n_quantiles_esperado
        ok_qmod  = len(q_mod) == n_quantiles_esperado

        # Validaciones estadísticas solo si hay datos
        adv = []
        if estado == "ok":
            if ok_qobs:
                adv += validar_cuantiles(q_obs, "q_obs")
            if ok_qmod:
                adv += validar_cuantiles(q_mod, "q_mod")
            if probs and len(probs) == n_quantiles_esperado:
                p = np.asarray(probs)
                if not (p[0] == 0.0 and p[-1] == 1.0):
                    adv.append("probs: no empieza en 0 o no termina en 1")
                if np.any(np.diff(p) < 0):
                    adv.append("probs: no monótona")

        if not (ok_probs and ok_qobs and ok_qmod) or adv or estado != "ok":
            todo_ok = False

        todas_advertencias += adv
        adv_txt = " | ".join(adv) if adv else "—"

        # Formato limpio de porcentajes
        def fmt_pct(v):
            try:
                return f"{float(v):>8.1f}"
            except Exception:
                return f"{str(v):>8}"

        print(
            f"  {str(mes_val):>4}  {str(estado):<22}  "
            f"{len(probs):>8}  {len(q_obs):>8}  {len(q_mod):>8}  "
            f"{str(n_obs):>6}  {str(n_mod):>6}  "
            f"{fmt_pct(p0_obs)}  {fmt_pct(p0_mod)}  "
            f"{adv_txt}"
        )

    # ── Resumen del archivo ──
    print(f"\n{'─'*65}")
    print("RESUMEN:")
    print(f"  Meses detectados : {meses_detectados}")

    meses_validos = [m for m in meses_detectados if m is not None]
    secuencia_ok  = (len(meses_validos) == 12 and
                     sorted([int(m) for m in meses_validos]) == list(range(1, 13)))
    print(f"  Secuencia meses  : {'OK (1..12)' if secuencia_ok else 'REVISAR'}")
    print(f"  Cuantiles/estado : {'OK' if todo_ok else 'REVISAR — ver columna Advertencias'}")

    if todas_advertencias:
        print("  Advertencias únicas encontradas:")
        for a in sorted(set(todas_advertencias)):
            print(f"    • {a}")
    else:
        print("  Sin advertencias estadísticas.")

    print(f"{'─'*65}")


# =====================================================
# 5. EJECUTAR INSPECCIÓN PARA TODOS LOS MODELOS × UMBRALES
# =====================================================
print("\n╔════════════════════════════════════════════════════════════╗")
print("║   INSPECCIÓN QDM DESDE DRIVE — DOBLE UMBRAL DE DÍA SECO  ║")
print("║   0.1 mm/día (u0p1)  |  0.5 mm/día (u0p5)                ║")
print(f"║   Directorio: {str(drive_base)[-45:]:<45} ║")
print("╚════════════════════════════════════════════════════════════╝")

for modelo in modelos:
    nombre = modelo.replace("-", "_")

    for tag_umbral, umbral_mm in UMBRALES.items():
        # Nombre del archivo según la convención del script de generación
        nombre_csv = f"QDM_{nombre}_{tag_umbral}_monthly.csv"
        ruta_csv   = drive_base / nombre_csv

        inspeccionar_csv_qdm(ruta_csv, modelo, tag_umbral, umbral_mm)

# ── Bonus: listar todos los archivos generados en el directorio ──
print("\n\n📁 Archivos disponibles en el directorio de salida:")
if drive_base.exists():
    archivos = sorted(drive_base.iterdir())
    for a in archivos:
        size_kb = a.stat().st_size / 1024
        print(f"  {a.name:<55}  {size_kb:>8.1f} KB")
else:
    print(f"  [AVISO] Directorio no encontrado: {drive_base}")

print("\n✓ Inspección completa.")



## 3.2 · Descarga CMIP6 — Precipitación (RAW)

Alineación estándar a 90 m sobre la grilla del proyecto.


In [ ]:
# ==============================================================
# DESCARGA CMIP6 PRECIPITACIÓN CRUDO (SIN CORRECCIÓN)
# Alineación Estándar 90m | Project001
# ==============================================================

import os, math
import ee, geemap
from google.colab import drive

# ==============================================================
# 1. MONTAR DRIVE Y CONFIGURAR RUTAS
# ==============================================================
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Project001"

# Referencia a la grilla estándar (Solo para asegurar el contexto)
# Archivo de referencia: GRID_WGS84_90m_template.tif
DIR_GRID = os.path.join(BASE, "GRID_TEMPLATE_WGS84")

# --- ESTRUCTURA DE CARPETAS SOLICITADA ---
# Carpeta principal de integración
DIR_CLIMA_TOTAL = os.path.join(BASE, "DATOS_CLIMATICOS_TOTALES")

# Subcarpeta donde se guardará la precipitación futura cruda
DIR_PPT_FUTURA = os.path.join(DIR_CLIMA_TOTAL, "PRECIPITACION_FUTURA_RAW")

# Carpeta de respaldo/proceso original (opcional, por coherencia con scripts previos)
DIR_CMIP6_RAW_BASE = os.path.join(BASE, "CMIP6_PR_RAW_WGS84")

# Crear la jerarquía de directorios
for d in [DIR_CLIMA_TOTAL, DIR_PPT_FUTURA, DIR_CMIP6_RAW_BASE]:
    os.makedirs(d, exist_ok=True)

# ==============================================================
# 2. PARÁMETROS TÉCNICOS (RESOLUCIÓN 90M)
# ==============================================================
AOI_ASSET     = "projects/mapas2025-473512/assets/AreaAporte"
PIXEL_M       = 90.0      # Resolución idéntica a la grilla plantilla
SOBREESCRIBIR = False

# Rango de años para el escenario futuro
ANIO_INI = 2015
ANIO_FIN = 2100

MODELOS   = ["MPI-ESM1-2-LR", "EC-Earth3-Veg-LR", "EC-Earth3"]
SCENARIOS = ["ssp245", "ssp585"]

# ==============================================================
# 3. INICIALIZAR EARTH ENGINE (PROYECTO ESPECÍFICO)
# ==============================================================
try:
    # Usando el ID de proyecto proporcionado
    ee.Initialize(project="computer-492420")
    print("GEE Inicializado con éxito.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="computer-492420")

# ==============================================================
# 4. CÁLCULO DE LA TRANSFORMACIÓN AFÍN (XFORM)
# ==============================================================
# Este bloque garantiza que la resolución sea EXACTAMENTE 90m
# y que los píxeles coincidan con tu GRID_WGS84_90m_template.tif

aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
bbox     = aoi_geom.bounds()

# Extraer coordenadas para el origen de la grilla
ring = ee.List(ee.List(bbox.coordinates().get(0)))
xmin = ee.Number(ee.List(ring.get(0)).get(0))
ymax = ee.Number(ee.List(ring.get(2)).get(1))

# Cálculo de grados por píxel (90m) basado en la latitud central
lat      = ee.Number(bbox.centroid(1).coordinates().get(1))
deg_y    = ee.Number(PIXEL_M / 111320.0)
deg_x    = ee.Number(PIXEL_M).divide(
    ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos())
)

# XFORM: Define [escala_x, cizalla_x, x_origen, cizalla_y, escala_y, y_origen]
# El valor deg_y es negativo porque en los rasters la coordenada Y disminuye hacia abajo
XFORM = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])

# ==============================================================
# 5. FUNCIÓN DE DESCARGA MENSUAL
# ==============================================================

def descargar_mes_cmip6_crudo(modelo, escenario, anio, mes, carpeta_destino):
    """
    Descarga, convierte unidades y fuerza la resolución a 90m.
    """
    mod_id = modelo.replace("-", "_")
    nombre_archivo = f"PR_RAW_{mod_id}_{escenario}_{anio}_{mes:02d}_WGS84_90m.tif"
    ruta_final = os.path.join(carpeta_destino, nombre_archivo)

    # Saltar descarga si el archivo ya existe
    if os.path.exists(ruta_final) and not SOBREESCRIBIR:
        return

    # 1. Cargar datos para el mes específico
    t_inicio = ee.Date.fromYMD(anio, mes, 1)
    col = (ee.ImageCollection("NASA/GDDP-CMIP6")
           .filter(ee.Filter.eq("model", modelo))
           .filter(ee.Filter.eq("scenario", escenario))
           .filterDate(t_inicio, t_inicio.advance(1, "month"))
           .select("pr"))

    # 2. Conversión: flujo de masa (kg/m2/s) a lámina diaria (mm/día)
    def a_mm_dia(img):
        return img.multiply(86400.0).copyProperties(img, ["system:time_start"])

    col_mm = col.map(a_mm_dia)

    # 3. Nombrar bandas según el día (YYYYMMdd) para facilitar lectura en modelos
    fechas = ee.List(col_mm.aggregate_array("system:time_start")).map(
        lambda t: ee.Date(t).format("YYYYMMdd")
    )

    # 4. Convertir colección a imagen multi-banda y forzar resolución/grilla
    # .reproject garantiza que el raster resultante tenga la grilla exacta de 90m
    img_export = (col_mm.toBands()
                  .rename(fechas)
                  .clip(bbox)
                  .reproject(crs="EPSG:4326", crsTransform=XFORM)
                  .toFloat())

    # 5. Exportar a Google Drive
    geemap.ee_export_image(
        img_export,
        filename=ruta_final,
        region=bbox,
        crs="EPSG:4326",
        crs_transform=XFORM,
        file_per_band=False,
        timeout=600
    )
    print(f"  Finalizado: {nombre_archivo}")

# ==============================================================
# 6. BUCLE DE PROCESAMIENTO
# ==============================================================

print(f">>> Iniciando descarga en: {DIR_PPT_FUTURA}")

for modelo in MODELOS:
    for escenario in SCENARIOS:
        print(f"\n--- Procesando Modelo: {modelo} | Escenario: {escenario} ---")

        # Crear subcarpeta interna por modelo y escenario
        sub_path = os.path.join(DIR_PPT_FUTURA, f"{modelo.replace('-','_')}_{escenario}")
        os.makedirs(sub_path, exist_ok=True)

        for anio in range(ANIO_INI, ANIO_FIN + 1):
            for mes in range(1, 13):
                try:
                    descargar_mes_cmip6_crudo(modelo, escenario, anio, mes, sub_path)
                except Exception as e:
                    print(f"  [ERROR] {anio}-{mes:02d}: {e}")

print("\n" + "="*60)
print("DESCARGA COMPLETADA Y ALINEADA A 90M")
print(f"Ubicación: {DIR_PPT_FUTURA}")
print("="*60)

In [ ]:
import os, math, json
import ee, geemap
from pathlib import Path
from tqdm.notebook import tqdm
from google.colab import drive

# ==============================================================
# 1. MONTAR DRIVE Y CONFIGURAR RUTAS
# ==============================================================
# Forzamos el montaje para asegurar que la conexión con Drive exista
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Definición de rutas usando Path (más robusto)
BASE = Path("/content/drive/MyDrive/Project001")
DIR_PPT_FUTURA = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_FUTURA_RAW"

# REGLA DE ORO: Crear la jerarquía de carpetas inmediatamente
DIR_PPT_FUTURA.mkdir(parents=True, exist_ok=True)

# Parámetros técnicos
AOI_ASSET = "projects/mapas2025-473512/assets/AreaAporte"
PIXEL_M    = 90.0
ANIO_INI   = 2015
ANIO_FIN   = 2100
MODELOS    = ["MPI-ESM1-2-LR", "EC-Earth3-Veg-LR", "EC-Earth3"]
SCENARIOS  = ["ssp245", "ssp585"]
N_BORRAR   = 2  # Cantidad de archivos a re-descargar al reiniciar

# ==============================================================
# 2. INICIALIZAR EARTH ENGINE
# ==============================================================
try:
    ee.Initialize(project="computer-492420")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="computer-492420")

# ==============================================================
# 3. FUNCIONES DE CONTROL (SISTEMA DE LOG)
# ==============================================================

def gestionar_log_y_reinicio(subcarpeta):
    """
    Lee el archivo JSON de control. Si encuentra datos de una sesión
    anterior, borra los 2 últimos archivos por seguridad.
    """
    ruta_log = subcarpeta / "_lista_descargados.json"

    if ruta_log.exists():
        with open(ruta_log, "r") as f:
            lista = json.load(f)

        # Lógica de borrado preventivo al reiniciar
        if len(lista) >= N_BORRAR:
            a_borrar = lista[-N_BORRAR:] # Los 2 últimos nombres
            lista = lista[:-N_BORRAR]    # Quitar del log

            for nombre in a_borrar:
                archivo_fisico = subcarpeta / nombre
                if archivo_fisico.exists():
                    archivo_fisico.unlink() # Borrar de Drive

            # Actualizar el log tras la limpieza
            with open(ruta_log, "w") as f:
                json.dump(lista, f)
        return lista
    return []

def guardar_progreso(subcarpeta, lista):
    """Guarda el nombre del archivo recién descargado en el JSON."""
    ruta_log = subcarpeta / "_lista_descargados.json"
    with open(ruta_log, "w") as f:
        json.dump(lista, f)

# ==============================================================
# 4. CONFIGURACIÓN GEOGRÁFICA (XFORM)
# ==============================================================
aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
bbox = aoi_geom.bounds()
ring = ee.List(ee.List(bbox.coordinates().get(0)))
xmin = ee.Number(ee.List(ring.get(0)).get(0))
ymax = ee.Number(ee.List(ring.get(2)).get(1))
lat = ee.Number(bbox.centroid(1).coordinates().get(1))
deg_y = ee.Number(PIXEL_M / 111320.0)
deg_x = ee.Number(PIXEL_M).divide(ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos()))
XFORM = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])

# ==============================================================
# 5. BUCLE DE PROCESAMIENTO
# ==============================================================

for modelo in MODELOS:
    for escenario in SCENARIOS:
        # Crear subcarpeta específica para el modelo y escenario
        mod_id = modelo.replace("-", "_")
        sub_path = DIR_PPT_FUTURA / f"{mod_id}_{escenario}"
        sub_path.mkdir(parents=True, exist_ok=True) # ASEGURA QUE EXISTA EN DRIVE

        # Cargar estado anterior y aplicar limpieza de los 2 últimos
        descargados = gestionar_log_y_reinicio(sub_path)

        # Identificar qué falta por descargar
        tareas_pendientes = []
        for anio in range(ANIO_INI, ANIO_FIN + 1):
            for mes in range(1, 13):
                nombre_tif = f"PR_RAW_{mod_id}_{escenario}_{anio}_{mes:02d}_WGS84_90m.tif"
                if nombre_tif not in descargados:
                    tareas_pendientes.append({'anio': anio, 'mes': mes, 'nombre': nombre_tif})

        if not tareas_pendientes:
            continue

        # BARRA DE PROGRESO: Controla la visualización sin ensuciar con prints
        pbar = tqdm(tareas_pendientes, desc=f"Descargando {mod_id} | {escenario}", unit="tif")

        for t in pbar:
            anio, mes, nombre = t['anio'], t['mes'], t['nombre']
            ruta_final = sub_path / nombre

            # Actualiza el texto de la barra con la fecha actual
            pbar.set_postfix(fecha=f"{anio}-{mes:02d}")

            try:
                # Procesamiento Earth Engine
                t_inicio = ee.Date.fromYMD(anio, mes, 1)
                col = (ee.ImageCollection("NASA/GDDP-CMIP6")
                       .filter(ee.Filter.eq("model", modelo))
                       .filter(ee.Filter.eq("scenario", escenario))
                       .filterDate(t_inicio, t_inicio.advance(1, "month"))
                       .select("pr"))

                col_mm = col.map(lambda img: img.multiply(86400.0).copyProperties(img, ["system:time_start"]))
                fechas = ee.List(col_mm.aggregate_array("system:time_start")).map(lambda t: ee.Date(t).format("YYYYMMdd"))

                img_export = (col_mm.toBands().rename(fechas).clip(bbox)
                              .reproject(crs="EPSG:4326", crsTransform=XFORM).toFloat())

                # Exportación (geemap lo enviará directamente a la ruta de Drive)
                geemap.ee_export_image(img_export, filename=str(ruta_final), region=bbox,
                                       crs="EPSG:4326", crs_transform=XFORM, timeout=600)

                # Si terminó sin error, anotar en el registro
                descargados.append(nombre)
                guardar_progreso(sub_path, descargados)

            except Exception as e:
                pbar.set_postfix(ERROR=f"{anio}-{mes}")
                continue

print("\n>>> PROCESO FINALIZADO. Revisa tu Drive en la carpeta Project001.")

In [ ]:
import os, math
import ee, geemap
from pathlib import Path
from tqdm.notebook import tqdm
from google.colab import drive

# ==============================================================
# 1. MONTAR DRIVE Y CONFIGURAR RUTAS
# ==============================================================
# Forzamos el montaje para asegurar que la conexión con Drive exista
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Definición de rutas usando Path (más robusto)
BASE = Path("/content/drive/MyDrive/Project001")
DIR_PPT_FUTURA = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_FUTURA_RAW"

# REGLA DE ORO: Crear la jerarquía de carpetas inmediatamente
DIR_PPT_FUTURA.mkdir(parents=True, exist_ok=True)

# Parámetros técnicos
AOI_ASSET = "projects/mapas2025-473512/assets/AreaAporte"
PIXEL_M    = 90.0
ANIO_INI   = 2100
ANIO_FIN   = 2100
MODELOS    = ["EC-Earth3"]
SCENARIOS  = ["ssp585"]

# ==============================================================
# 2. INICIALIZAR EARTH ENGINE
# ==============================================================
try:
    ee.Initialize(project="computer-492420")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="computer-492420")

# ==============================================================
# 4. CONFIGURACIÓN GEOGRÁFICA (XFORM)
# ==============================================================
aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
bbox = aoi_geom.bounds()
ring = ee.List(ee.List(bbox.coordinates().get(0)))
xmin = ee.Number(ee.List(ring.get(0)).get(0))
ymax = ee.Number(ee.List(ring.get(2)).get(1))
lat = ee.Number(bbox.centroid(1).coordinates().get(1))
deg_y = ee.Number(PIXEL_M / 111320.0)
deg_x = ee.Number(PIXEL_M).divide(ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos()))
XFORM = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])

# ==============================================================
# 5. BUCLE DE PROCESAMIENTO
# ==============================================================

for modelo in MODELOS:
    for escenario in SCENARIOS:
        # Crear subcarpeta específica para el modelo y escenario
        mod_id = modelo.replace("-", "_")
        sub_path = DIR_PPT_FUTURA / f"{mod_id}_{escenario}"
        sub_path.mkdir(parents=True, exist_ok=True)

        # Solo este archivo exacto
        tareas_pendientes = []
        for anio in range(ANIO_INI, ANIO_FIN + 1):
            for mes in [12]:
                nombre_tif = f"PR_RAW_{mod_id}_{escenario}_{anio}_{mes:02d}_WGS84_90m.tif"
                tareas_pendientes.append({'anio': anio, 'mes': mes, 'nombre': nombre_tif})

        if not tareas_pendientes:
            continue

        pbar = tqdm(tareas_pendientes, desc=f"Descargando {mod_id} | {escenario}", unit="tif")

        for t in pbar:
            anio, mes, nombre = t['anio'], t['mes'], t['nombre']
            ruta_final = sub_path / nombre

            pbar.set_postfix(fecha=f"{anio}-{mes:02d}")

            try:
                print(f"\nIntentando descargar: {nombre}")

                # Procesamiento Earth Engine
                t_inicio = ee.Date.fromYMD(anio, mes, 1)
                col = (ee.ImageCollection("NASA/GDDP-CMIP6")
                       .filter(ee.Filter.eq("model", modelo))
                       .filter(ee.Filter.eq("scenario", escenario))
                       .filterDate(t_inicio, t_inicio.advance(1, "month"))
                       .select("pr"))

                col_mm = col.map(lambda img: img.multiply(86400.0).copyProperties(img, ["system:time_start"]))
                fechas = ee.List(col_mm.aggregate_array("system:time_start")).map(lambda t: ee.Date(t).format("YYYYMMdd"))

                img_export = (col_mm.toBands().rename(fechas).clip(bbox)
                              .reproject(crs="EPSG:4326", crsTransform=XFORM).toFloat())

                # Exportación
                geemap.ee_export_image(
                    img_export,
                    filename=str(ruta_final),
                    region=bbox,
                    crs="EPSG:4326",
                    crs_transform=XFORM,
                    timeout=600
                )

                # Verificación real del archivo
                if ruta_final.exists() and ruta_final.stat().st_size > 0:
                    print(f"[OK] Archivo creado correctamente: {ruta_final}")
                else:
                    print(f"[FALLO] No se creó el archivo: {ruta_final}")

            except Exception as e:
                print(f"[ERROR] {anio}-{mes:02d}: {e}")
                pbar.set_postfix(ERROR=f"{anio}-{mes}")
                continue

print("\n>>> PROCESO FINALIZADO. Revisa tu Drive en la carpeta Project001.")

## 3.3 · Descarga CMIP6 — Temperatura (tasmax / tasmin RAW)


In [ ]:
import os, math, json, sys, contextlib
import ee, geemap
from pathlib import Path
from tqdm.notebook import tqdm
from google.colab import drive

# ==============================================================
# 1. MONTAR DRIVE Y CONFIGURAR RUTAS
# ==============================================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE = Path("/content/drive/MyDrive/Project001")
DIR_CLIMA_TOTAL = BASE / "DATOS_CLIMATICOS_TOTALES"
DIR_TMAX_FUTURA = DIR_CLIMA_TOTAL / "TEMPERATURA_MAX_FUTURA_RAW"
DIR_TMIN_FUTURA = DIR_CLIMA_TOTAL / "TEMPERATURA_MIN_FUTURA_RAW"

for d in [DIR_TMAX_FUTURA, DIR_TMIN_FUTURA]:
    d.mkdir(parents=True, exist_ok=True)

# ==============================================================
# 2. PARÁMETROS TÉCNICOS
# ==============================================================
AOI_ASSET = "projects/mapas2025-473512/assets/AreaAporte"
PIXEL_M    = 90.0
ANIO_INI   = 2015
ANIO_FIN   = 2100
N_BORRAR   = 2

MODELOS   = ["MPI-ESM1-2-LR", "EC-Earth3-Veg-LR", "EC-Earth3"]
SCENARIOS = ["ssp245", "ssp585"]

# ==============================================================
# 3. FUNCIONES DE CONTROL (REGISTRO TXT)
# ==============================================================

def gestionar_registro_temperaturas(sub_path_tmax, sub_path_tmin):
    ruta_reg = sub_path_tmax / "_registro_temperaturas.txt"
    descargados = []

    if ruta_reg.exists():
        with open(ruta_reg, "r") as f:
            descargados = [line.strip() for line in f.readlines() if line.strip()]

        if len(descargados) >= N_BORRAR:
            a_borrar = descargados[-N_BORRAR:]
            descargados = descargados[:-N_BORRAR]
            for nombre_base in a_borrar:
                f_tmax = sub_path_tmax / f"TMAX_RAW_{nombre_base}.tif"
                f_tmin = sub_path_tmin / f"TMIN_RAW_{nombre_base}.tif"
                if f_tmax.exists(): f_tmax.unlink()
                if f_tmin.exists(): f_tmin.unlink()
            with open(ruta_reg, "w") as f:
                f.write("\n".join(descargados) + "\n")
    else:
        ruta_reg.touch()
    return descargados

def registrar_descarga(sub_path_tmax, nombre_base):
    ruta_reg = sub_path_tmax / "_registro_temperaturas.txt"
    with open(ruta_reg, "a") as f:
        f.write(nombre_base + "\n")
        os.fsync(f.fileno())

# ==============================================================
# 4. INICIALIZAR GEE
# ==============================================================
try:
    ee.Initialize(project="computer-492420")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="computer-492420")

aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
bbox = aoi_geom.bounds()
ring = ee.List(ee.List(bbox.coordinates().get(0)))
xmin = ee.Number(ee.List(ring.get(0)).get(0))
ymax = ee.Number(ee.List(ring.get(2)).get(1))
lat = ee.Number(bbox.centroid(1).coordinates().get(1))
deg_y = ee.Number(PIXEL_M / 111320.0)
deg_x = ee.Number(PIXEL_M).divide(ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos()))
XFORM = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])

# ==============================================================
# 5. BUCLE DE PROCESAMIENTO SILENCIOSO
# ==============================================================

print(">>> Iniciando descarga de Temperaturas. Solo se mostrarán las barras de progreso.")

for modelo in MODELOS:
    for escenario in SCENARIOS:
        mod_id = modelo.replace("-", "_")
        sub_tmax = DIR_TMAX_FUTURA / f"{mod_id}_{escenario}"
        sub_tmin = DIR_TMIN_FUTURA / f"{mod_id}_{escenario}"
        sub_tmax.mkdir(parents=True, exist_ok=True)
        sub_tmin.mkdir(parents=True, exist_ok=True)

        registrados = gestionar_registro_temperaturas(sub_tmax, sub_tmin)

        tareas = []
        for anio in range(ANIO_INI, ANIO_FIN + 1):
            for mes in range(1, 13):
                id_mes = f"{mod_id}_{escenario}_{anio}_{mes:02d}_WGS84_90m"
                if id_mes not in registrados:
                    tareas.append({'anio': anio, 'mes': mes, 'id_mes': id_mes})

        if not tareas: continue

        # Barra de progreso principal
        pbar = tqdm(tareas, desc=f"🌡️ {mod_id} | {escenario}", unit="mes")

        for t in pbar:
            anio, mes, id_mes = t['anio'], t['mes'], t['id_mes']
            pbar.set_postfix(fecha=f"{anio}-{mes:02d}")

            try:
                t_inicio = ee.Date.fromYMD(anio, mes, 1)

                # RECURSO CRÍTICO: Redirigir la salida a "devnull" para silenciar geemap
                with open(os.devnull, 'w') as fnull:
                    with contextlib.redirect_stdout(fnull):

                        # Procesar TMAX y TMIN
                        for var, prefix, carpeta in [('tasmax', 'TMAX', sub_tmax), ('tasmin', 'TMIN', sub_tmin)]:
                            col = (ee.ImageCollection("NASA/GDDP-CMIP6")
                                   .filter(ee.Filter.eq("model", modelo))
                                   .filter(ee.Filter.eq("scenario", escenario))
                                   .filterDate(t_inicio, t_inicio.advance(1, "month"))
                                   .select(var))

                            col_celsius = col.map(lambda img: img.subtract(273.15).copyProperties(img, ["system:time_start"]))
                            fechas = ee.List(col_celsius.aggregate_array("system:time_start")).map(
                                lambda t: ee.Date(t).format("YYYYMMdd")
                            )

                            img_export = (col_celsius.toBands().rename(fechas).clip(bbox)
                                          .reproject(crs="EPSG:4326", crsTransform=XFORM).toFloat())

                            ruta_final = carpeta / f"{prefix}_RAW_{id_mes}.tif"

                            # Esta función es la que genera los mensajes "Generating URL...", etc.
                            geemap.ee_export_image(img_export, filename=str(ruta_final), region=bbox,
                                                   crs="EPSG:4326", crs_transform=XFORM, timeout=600)

                # Una vez fuera del bloqueo de silencio, registramos el éxito
                registrar_descarga(sub_tmax, id_mes)

            except Exception as e:
                # Los errores sí se mostrarán en la barra para que sepas si algo falló
                pbar.set_postfix(ERR=f"{anio}-{mes}")
                continue

print("\n>>> DESCARGA COMPLETADA.")

In [ ]:
# ==============================================================
# DESCARGA CMIP6 TEMPERATURAS CRUDO (SIN CORRECCIÓN)
# Alineación Estándar 90m | Project001
# ==============================================================

import os, math
import ee, geemap
from google.colab import drive

# ==============================================================
# 1. MONTAR DRIVE Y CONFIGURAR RUTAS
# ==============================================================
#drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Project001"

# --- ESTRUCTURA DE CARPETAS ---
DIR_CLIMA_TOTAL = os.path.join(BASE, "DATOS_CLIMATICOS_TOTALES")

# Subcarpetas para Temperatura Máxima y Mínima
DIR_TMAX_FUTURA = os.path.join(DIR_CLIMA_TOTAL, "TEMPERATURA_MAX_FUTURA_RAW")
DIR_TMIN_FUTURA = os.path.join(DIR_CLIMA_TOTAL, "TEMPERATURA_MIN_FUTURA_RAW")

# Crear directorios
for d in [DIR_CLIMA_TOTAL, DIR_TMAX_FUTURA, DIR_TMIN_FUTURA]:
    os.makedirs(d, exist_ok=True)

# ==============================================================
# 2. PARÁMETROS TÉCNICOS (RESOLUCIÓN 90M)
# ==============================================================
AOI_ASSET     = "projects/mapas2025-473512/assets/AreaAporte"
PIXEL_M       = 90.0
SOBREESCRIBIR = False

ANIO_INI = 2015
ANIO_FIN = 2100

MODELOS   = ["MPI-ESM1-2-LR", "EC-Earth3-Veg-LR", "EC-Earth3"]
SCENARIOS = ["ssp245", "ssp585"]

# ==============================================================
# 3. INICIALIZAR EARTH ENGINE
# ==============================================================
try:
    # Usando el ID de proyecto proporcionado
    ee.Initialize(project="computer-492420")
    print("GEE Inicializado con éxito.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="computer-492420")

# ==============================================================
# 4. CÁLCULO DE LA TRANSFORMACIÓN AFÍN (XFORM)
# ==============================================================
aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
bbox     = aoi_geom.bounds()
ring     = ee.List(ee.List(bbox.coordinates().get(0)))
xmin     = ee.Number(ee.List(ring.get(0)).get(0))
ymax     = ee.Number(ee.List(ring.get(2)).get(1))
lat      = ee.Number(bbox.centroid(1).coordinates().get(1))

deg_y    = ee.Number(PIXEL_M / 111320.0)
deg_x    = ee.Number(PIXEL_M).divide(
    ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos())
)

XFORM = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])

# ==============================================================
# 5. FUNCIÓN DE DESCARGA MENSUAL (TEMPERATURA)
# ==============================================================

def descargar_mes_temp_crudo(modelo, escenario, anio, mes, variable, carpeta_base):
    """
    Descarga Temperatura, convierte de Kelvin a Celsius y alinea a 90m.
    variable: 'tasmax' o 'tasmin'
    """
    mod_id = modelo.replace("-", "_")
    prefix = "TMAX" if variable == "tasmax" else "TMIN"
    nombre_archivo = f"{prefix}_RAW_{mod_id}_{escenario}_{anio}_{mes:02d}_WGS84_90m.tif"

    # Organizar por modelo/escenario dentro de la carpeta base
    sub_carpeta = os.path.join(carpeta_base, f"{mod_id}_{escenario}")
    if not os.path.exists(sub_carpeta):
        os.makedirs(sub_carpeta, exist_ok=True)

    ruta_final = os.path.join(sub_carpeta, nombre_archivo)

    if os.path.exists(ruta_final) and not SOBREESCRIBIR:
        return

    # 1. Cargar datos
    t_inicio = ee.Date.fromYMD(anio, mes, 1)
    col = (ee.ImageCollection("NASA/GDDP-CMIP6")
           .filter(ee.Filter.eq("model", modelo))
           .filter(ee.Filter.eq("scenario", escenario))
           .filterDate(t_inicio, t_inicio.advance(1, "month"))
           .select(variable))

    # 2. Conversión: Kelvin a Celsius (K - 273.15)
    def a_celsius(img):
        return img.subtract(273.15).copyProperties(img, ["system:time_start"])

    col_celsius = col.map(a_celsius)

    # 3. Nombres de bandas (YYYYMMdd)
    fechas = ee.List(col_celsius.aggregate_array("system:time_start")).map(
        lambda t: ee.Date(t).format("YYYYMMdd")
    )

    # 4. Apilar y Reproyectar a 90m
    img_export = (col_celsius.toBands()
                  .rename(fechas)
                  .clip(bbox)
                  .reproject(crs="EPSG:4326", crsTransform=XFORM)
                  .toFloat())

    # 5. Exportar
    geemap.ee_export_image(
        img_export,
        filename=ruta_final,
        region=bbox,
        crs="EPSG:4326",
        crs_transform=XFORM,
        file_per_band=False,
        timeout=600
    )
    print(f"  Finalizado: {nombre_archivo}")

# ==============================================================
# 6. BUCLE DE PROCESAMIENTO
# ==============================================================

for modelo in MODELOS:
    for escenario in SCENARIOS:
        print(f"\n>>> Procesando Modelo: {modelo} | Escenario: {escenario}")

        for anio in range(ANIO_INI, ANIO_FIN + 1):
            for mes in range(1, 13):
                try:
                    # Descargar Máxima
                    descargar_mes_temp_crudo(modelo, escenario, anio, mes, 'tasmax', DIR_TMAX_FUTURA)
                    # Descargar Mínima
                    descargar_mes_temp_crudo(modelo, escenario, anio, mes, 'tasmin', DIR_TMIN_FUTURA)
                except Exception as e:
                    print(f"  [ERROR] {anio}-{mes:02d}: {e}")

print("\n" + "="*60)
print("DESCARGA DE TEMPERATURAS COMPLETADA")
print(f"Máximas en: {DIR_TMAX_FUTURA}")
print(f"Mínimas en: {DIR_TMIN_FUTURA}")
print("="*60)

### 3.3.1 · Descarga especial de archivos faltantes

Descarga puntual del archivo EC-Earth3 · ssp585 · 2100-12 · PR mensual.


In [ ]:
# ==============================================================
# DESCARGA ESPECIAL SOLO DEL ARCHIVO FALTANTE
# EC-Earth3 | ssp585 | 2100-12 | PR mensual
# ==============================================================

import os
import time
import requests
from pathlib import Path

# =====================================================
# 1. MONTAR DRIVE (solo si estás en Colab)
# =====================================================
try:
    from google.colab import drive
    if not Path("/content/drive").exists():
        Path("/content/drive").mkdir(parents=True, exist_ok=True)
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"[INFO] Drive ya montado o no estás en Colab: {e}")

# =====================================================
# 2. IMPORTAR E INICIALIZAR EARTH ENGINE
# =====================================================
import ee

try:
    ee.Initialize(project="computer-492420")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="computer-492420")

# =====================================================
# 3. CONFIGURACIÓN
# =====================================================
BASE = Path("/content/drive/MyDrive/Project001")

DIR_SALIDA = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_FUTURA_RAW" / "EC_Earth3_ssp585"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

NOMBRE_ARCHIVO = "PR_RAW_EC_Earth3_ssp585_2100_12_WGS84_90m.tif"
RUTA_SALIDA = DIR_SALIDA / NOMBRE_ARCHIVO
RUTA_TEMP = DIR_SALIDA / f"{NOMBRE_ARCHIVO}.part"

ROI_ASSET = "projects/mapas2025-473512/assets/AreaAporte"

MODELO = "EC-Earth3"
ESCENARIO = "ssp585"
VARIABLE = "pr"
ANIO = 2100
MES = 12

CRS = "EPSG:4326"
SCALE = 90

MAX_INTENTOS = 5
ESPERA_SEGUNDOS = 10
TIMEOUT_DESCARGA = 300

FORZAR_REDESCARGA = True  # pon False si no quieres reemplazar si ya existe


# =====================================================
# 4. ROI
# =====================================================
roi_fc = ee.FeatureCollection(ROI_ASSET)
roi_geom = roi_fc.geometry()

# =====================================================
# 5. FUNCIONES
# =====================================================

def verificar_disponibilidad_mes(modelo, escenario, anio, mes, variable="pr"):
    start = ee.Date.fromYMD(anio, mes, 1)
    end = start.advance(1, "month")

    col = (
        ee.ImageCollection("NASA/GDDP-CMIP6")
        .filter(ee.Filter.eq("model", modelo))
        .filter(ee.Filter.eq("scenario", escenario))
        .filterDate(start, end)
        .select(variable)
    )

    n = col.size().getInfo()
    print(f"[CHECK] {modelo} | {escenario} | {anio}-{mes:02d} | {variable} -> {n} imagen(es)")

    if n > 0:
        try:
            fechas = col.aggregate_array("system:time_start").getInfo()
            print(f"[CHECK] Primeras fechas disponibles:")
            for f in fechas[:5]:
                print("   -", ee.Date(f).format("YYYY-MM-dd").getInfo())
        except Exception as e:
            print(f"[WARN] No se pudieron listar fechas: {e}")

    return n


def construir_imagen_pr_mensual(modelo, escenario, anio, mes):
    """
    Asume que 'pr' en GDDP-CMIP6 está en kg m-2 s-1 (equiv. mm/s).
    Para acumulado mensual en mm/mes:
        pr_dia_mm = pr * 86400
        pr_mes_mm = suma diaria del mes
    """
    start = ee.Date.fromYMD(anio, mes, 1)
    end = start.advance(1, "month")

    col = (
        ee.ImageCollection("NASA/GDDP-CMIP6")
        .filter(ee.Filter.eq("model", modelo))
        .filter(ee.Filter.eq("scenario", escenario))
        .filterDate(start, end)
        .select("pr")
    )

    n = col.size().getInfo()
    if n == 0:
        raise RuntimeError(f"No hay imágenes para {modelo} | {escenario} | {anio}-{mes:02d}")

    # Convertir a mm/día y luego acumular el mes
    col_mm_dia = col.map(lambda img: img.multiply(86400).copyProperties(img, img.propertyNames()))
    img_mes = (
        col_mm_dia
        .sum()
        .rename("pr")
        .clip(roi_geom)
        .set({
            "modelo": modelo,
            "escenario": escenario,
            "anio": anio,
            "mes": mes,
            "variable": "pr",
            "unidad": "mm/mes"
        })
    )

    return img_mes


def descargar_imagen_ee(image, out_path, temp_path, scale=90, crs="EPSG:4326", max_intentos=5):
    """
    Descarga robusta usando getDownloadURL + requests.
    Guarda primero en .part y luego renombra a .tif
    """
    region = roi_geom.bounds(1).getInfo()["coordinates"]

    params = {
        "name": out_path.stem,
        "scale": scale,
        "crs": crs,
        "region": region,
        "format": "GEO_TIFF",
        "filePerBand": False
    }

    for intento in range(1, max_intentos + 1):
        try:
            print(f"\n[INTENTO {intento}/{max_intentos}] Generando URL...")
            url = image.getDownloadURL(params)
            print("[OK] URL generada")

            if temp_path.exists():
                temp_path.unlink()

            print("[DESCARGA] Descargando raster...")
            with requests.get(url, stream=True, timeout=TIMEOUT_DESCARGA) as r:
                r.raise_for_status()

                with open(temp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            # Validación simple de tamaño
            if not temp_path.exists() or temp_path.stat().st_size == 0:
                raise RuntimeError("El archivo descargado quedó vacío.")

            # Si ya existe el final, se reemplaza
            if out_path.exists():
                out_path.unlink()

            temp_path.rename(out_path)

            print(f"[OK] Descarga completada: {out_path}")
            print(f"[OK] Tamaño archivo: {out_path.stat().st_size / (1024**2):.2f} MB")
            return True

        except Exception as e:
            print(f"[ERROR] Falló intento {intento}: {e}")

            if temp_path.exists():
                try:
                    temp_path.unlink()
                except:
                    pass

            if intento < max_intentos:
                print(f"[REINTENTO] Esperando {ESPERA_SEGUNDOS} segundos...")
                time.sleep(ESPERA_SEGUNDOS)
            else:
                print("[FALLO FINAL] No se pudo descargar el archivo.")
                return False


# =====================================================
# 6. PROCESO
# =====================================================
print("=" * 70)
print("DESCARGA ESPECIAL DEL ARCHIVO FALTANTE")
print("=" * 70)
print(f"Modelo     : {MODELO}")
print(f"Escenario  : {ESCENARIO}")
print(f"Año-Mes    : {ANIO}-{MES:02d}")
print(f"Salida     : {RUTA_SALIDA}")
print("=" * 70)

# Si ya existe y no quieres reemplazar
if RUTA_SALIDA.exists() and not FORZAR_REDESCARGA:
    print(f"[INFO] El archivo ya existe y FORZAR_REDESCARGA=False:")
    print(RUTA_SALIDA)
else:
    if RUTA_SALIDA.exists() and FORZAR_REDESCARGA:
        print("[INFO] Ya existía el archivo. Se reemplazará.")
        RUTA_SALIDA.unlink()

    n = verificar_disponibilidad_mes(MODELO, ESCENARIO, ANIO, MES, VARIABLE)

    if n == 0:
        raise RuntimeError(f"No hay datos para {MODELO} {ESCENARIO} {ANIO}-{MES:02d}")
    else:
        img_mes = construir_imagen_pr_mensual(MODELO, ESCENARIO, ANIO, MES)
        exito = descargar_imagen_ee(
            image=img_mes,
            out_path=RUTA_SALIDA,
            temp_path=RUTA_TEMP,
            scale=SCALE,
            crs=CRS,
            max_intentos=MAX_INTENTOS
        )

        if exito:
            print("\n[FINALIZADO] El archivo faltante se descargó correctamente.")
        else:
            print("\n[ATENCIÓN] La descarga no se completó.")

## 3.4 · Tablas QDM mensuales — Temperatura


In [ ]:
import ee
import pandas as pd
import numpy as np
from pathlib import Path
import json

# =====================================================
# 1. MONTAR GOOGLE DRIVE Y CONFIGURAR RUTAS
# =====================================================


drive_base = Path("/content/drive/MyDrive/Project001")
drive_base.mkdir(parents=True, exist_ok=True)

output_dir = drive_base / "salidas_qdm_temperatura_mensual"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Archivos se guardarán en: {output_dir}")

# =====================================================
# 2. INICIALIZAR EARTH ENGINE
# =====================================================
try:
    ee.Initialize(project='computer-492420')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='computer-492420')

# =====================================================
# 3. PARÁMETROS GENERALES
# =====================================================
roi_fc   = ee.FeatureCollection("projects/mapas2025-473512/assets/AreaAporte")
roi_geom = roi_fc.geometry()

start_date = "1985-01-01"
end_date   = "2014-12-31"

modelos = [
    "MPI-ESM1-2-LR",
    "EC-Earth3-Veg-LR",
    "EC-Earth3"
]

scale_era5  = 5000
scale_cmip6 = 5000
sigma_min   = 0.01   # evita desviación estándar cero

# Variables de temperatura a procesar
VARIABLES_TEMP = [
    {
        "nombre_variable": "tasmax",
        "col_obs":         "tmax_obs",
        "col_mod":         "tmax_mod",
        "era5_var":        "temperature_2m_max",
        "cmip6_var":       "tasmax"
    },
    {
        "nombre_variable": "tasmin",
        "col_obs":         "tmin_obs",
        "col_mod":         "tmin_mod",
        "era5_var":        "temperature_2m_min",
        "cmip6_var":       "tasmin"
    }
]

min_muestras_mes = 20

# =====================================================
# 4. FUNCIONES AUXILIARES
# =====================================================

def extraer_serie_temporal(collection, var_name, escala, start_date, end_date,
                            convertir_a_celsius=False):
    """
    Extrae serie temporal diaria promediada sobre el ROI.
    Si convertir_a_celsius=True convierte de Kelvin → Celsius (resta 273.15).
    """
    def reduce_region(img):
        val = img.reduceRegion(
            reducer   = ee.Reducer.mean(),
            geometry  = roi_geom,
            scale     = escala,
            maxPixels = 1e9
        ).get(var_name)

        if convertir_a_celsius:
            val = ee.Number(val).subtract(273.15)

        return img.set("valor", val).set("date", img.date().format("YYYY-MM-dd"))

    col     = collection.filterDate(start_date, end_date).map(reduce_region)
    fechas  = col.aggregate_array("date").getInfo()
    valores = col.aggregate_array("valor").getInfo()

    df = pd.DataFrame({
        "date" : pd.to_datetime(fechas,  errors="coerce"),
        "value": pd.to_numeric(valores, errors="coerce")
    })
    return df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)


def limpiar_serie_temperatura(arr):
    """Limpia una serie de temperatura: convierte a float y elimina no-finitos."""
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]


def asegurar_sigma(sigma, sigma_min=0.01):
    sigma = float(sigma)
    if not np.isfinite(sigma) or sigma < sigma_min:
        return sigma_min
    return sigma


def construir_tabla_normal_mensual(df, col_obs, col_mod,
                                   min_muestras_mes=20, sigma_min=0.01):
    """
    Genera tabla mensual con parámetros de distribución normal para QDM de temperatura.
    Campos: mes, mu_obs_hist, sigma_obs_hist, mu_mod_hist, sigma_mod_hist, n_obs, n_mod, estado.
    """
    rows = []
    for mes in range(1, 13):
        sub = df[df["mes"] == mes].copy()
        obs = limpiar_serie_temperatura(sub[col_obs].to_numpy(dtype=float))
        mod = limpiar_serie_temperatura(sub[col_mod].to_numpy(dtype=float))

        if len(obs) < min_muestras_mes or len(mod) < min_muestras_mes:
            rows.append({
                "mes":            mes,
                "mu_obs_hist":    np.nan,
                "sigma_obs_hist": np.nan,
                "mu_mod_hist":    np.nan,
                "sigma_mod_hist": np.nan,
                "n_obs":          int(len(obs)),
                "n_mod":          int(len(mod)),
                "estado":         "insuficiente_muestra"
            })
            continue

        rows.append({
            "mes":            mes,
            "mu_obs_hist":    float(np.mean(obs)),
            "sigma_obs_hist": asegurar_sigma(np.std(obs, ddof=1), sigma_min=sigma_min),
            "mu_mod_hist":    float(np.mean(mod)),
            "sigma_mod_hist": asegurar_sigma(np.std(mod, ddof=1), sigma_min=sigma_min),
            "n_obs":          int(len(obs)),
            "n_mod":          int(len(mod)),
            "estado":         "ok"
        })

    return pd.DataFrame(rows)


def validar_tabla_normal(tabla_df):
    """
    Audita la tabla mensual de parámetros normales.
    Verifica que mu y sigma sean finitos y sigma > 0.
    """
    resultados = []
    for _, row in tabla_df.iterrows():
        mes    = int(row["mes"])
        estado = row["estado"]

        if estado != "ok":
            resultados.append({"mes": mes, "valido": False, "motivo": estado})
            continue

        motivos = []
        for campo in ["mu_obs_hist", "sigma_obs_hist", "mu_mod_hist", "sigma_mod_hist"]:
            if not np.isfinite(row[campo]):
                motivos.append(f"{campo}_no_finito")
        if row["sigma_obs_hist"] <= 0:
            motivos.append("sigma_obs_cero_o_negativo")
        if row["sigma_mod_hist"] <= 0:
            motivos.append("sigma_mod_cero_o_negativo")

        resultados.append({
            "mes":    mes,
            "valido": len(motivos) == 0,
            "motivo": ";".join(motivos) if motivos else "ok"
        })

    return pd.DataFrame(resultados)


def resumen_serie(df, col):
    """Estadísticas básicas de una columna numérica."""
    x = pd.to_numeric(df[col], errors="coerce")
    return {
        "n":     int(x.notna().sum()),
        "min":   float(x.min()),
        "max":   float(x.max()),
        "media": float(x.mean()),
        "p05":   float(x.quantile(0.05)),
        "p95":   float(x.quantile(0.95))
    }


def guardar_metadata_modelo(modelo, variable, df_hist, out_dir,
                             min_muestras_mes, sigma_min):
    meta = pd.DataFrame([{
        "modelo":           modelo,
        "variable":         variable,
        "metodo":           "QDM_normal_mensual",
        "fecha_inicio":     df_hist["date"].min().strftime("%Y-%m-%d"),
        "fecha_fin":        df_hist["date"].max().strftime("%Y-%m-%d"),
        "n_registros":      int(len(df_hist)),
        "observado":        "ERA5-Land",
        "modelo_fuente":    "NASA/GDDP-CMIP6",
        "unidad_final":     "C",
        "distribucion":     "normal",
        "min_muestras_mes": min_muestras_mes,
        "sigma_min":        sigma_min,
        "nota": (
            "Archivo preparado para corrección futura de temperatura con QDM mensual. "
            "Parámetros: media y desviación estándar mensuales de ERA5-Land y CMIP6. "
            "CSV guardado sin BOM (utf-8)."
        )
    }])
    nombre   = modelo.replace("-", "_")
    out_meta = out_dir / f"METADATA_{nombre}_{variable}.csv"
    meta.to_csv(out_meta, index=False, encoding="utf-8")
    print(f"  Guardado: {out_meta.name}")


# =====================================================
# 5. BASE CMIP6 (filtrado temporal previo al bucle)
# =====================================================
cmip6_base = ee.ImageCollection("NASA/GDDP-CMIP6").filterDate(start_date, end_date)

# =====================================================
# 6. BUCLE PRINCIPAL: variable × modelo
# =====================================================
print(f"\n[1/3] Procesando {len(VARIABLES_TEMP)} variables × {len(modelos)} modelos...\n")

for var_cfg in VARIABLES_TEMP:
    nombre_variable = var_cfg["nombre_variable"]
    col_obs         = var_cfg["col_obs"]
    col_mod         = var_cfg["col_mod"]
    era5_var        = var_cfg["era5_var"]
    cmip6_var       = var_cfg["cmip6_var"]

    print("=" * 70)
    print(f"VARIABLE: {nombre_variable}  |  ERA5: {era5_var}  |  CMIP6: {cmip6_var}")
    print("=" * 70)

    # ── 6a. Extraer observado ERA5-Land (una sola vez por variable) ──
    print(f"\n  Extrayendo ERA5-Land histórico para {nombre_variable}...")
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(era5_var)

    df_obs = extraer_serie_temporal(
        era5, era5_var, scale_era5,
        start_date, end_date,
        convertir_a_celsius=True
    ).rename(columns={"value": col_obs})

    out_obs = output_dir / f"OBS_ERA5_{nombre_variable}_historico.csv"
    df_obs.to_csv(out_obs, index=False, encoding="utf-8")
    print(f"  Guardado: {out_obs.name}")

    # ── 6b. Iterar sobre cada modelo ──
    for modelo in modelos:
        print(f"\n  ══ Modelo: {modelo} ══")
        nombre = modelo.replace("-", "_")

        # Extraer serie CMIP6 del modelo
        cmip6_mod = (
            cmip6_base
            .filter(ee.Filter.eq("model",    modelo))
            .filter(ee.Filter.eq("scenario", "historical"))
            .select(cmip6_var)
        )

        df_mod = extraer_serie_temporal(
            cmip6_mod, cmip6_var, scale_cmip6,
            start_date, end_date,
            convertir_a_celsius=True
        ).rename(columns={"value": col_mod})

        out_mod = output_dir / f"MOD_{nombre}_{nombre_variable}_historico.csv"
        df_mod.to_csv(out_mod, index=False, encoding="utf-8")
        print(f"    Guardado serie modelo: {out_mod.name}")

        # Emparejar observado + modelo por fecha (inner join)
        df_hist = pd.merge(df_obs, df_mod, on="date", how="inner").dropna().copy()
        df_hist["mes"]  = df_hist["date"].dt.month.astype(int)
        df_hist["anio"] = df_hist["date"].dt.year.astype(int)
        df_hist["dia"]  = df_hist["date"].dt.day.astype(int)

        out_hist = output_dir / f"HIST_{nombre}_{nombre_variable}_emparejado.csv"
        df_hist.to_csv(out_hist, index=False, encoding="utf-8")
        print(f"    Guardado histórico emparejado: {out_hist.name}")

        # Construir tabla de parámetros normales mensuales
        tabla_df = construir_tabla_normal_mensual(
            df_hist,
            col_obs          = col_obs,
            col_mod          = col_mod,
            min_muestras_mes = min_muestras_mes,
            sigma_min        = sigma_min
        )

        out_tabla = output_dir / f"QDM_{nombre}_{nombre_variable}_monthly_normal.csv"
        tabla_df.to_csv(out_tabla, index=False, encoding="utf-8")
        print(f"    Guardado tabla QDM normal: {out_tabla.name}")

        # Validar tabla mensual
        valid_df = validar_tabla_normal(tabla_df)
        out_valid = output_dir / f"VALIDACION_QDM_{nombre}_{nombre_variable}.csv"
        valid_df.to_csv(out_valid, index=False, encoding="utf-8")
        print(f"    Guardado validación: {out_valid.name}")

        # Resumen estadístico en JSON
        resumen = {
            "modelo":          modelo,
            "variable":        nombre_variable,
            "historico_obs":   resumen_serie(df_hist, col_obs),
            "historico_mod":   resumen_serie(df_hist, col_mod),
            "meses_validos":   int(valid_df["valido"].sum()),
            "meses_invalidos": int((~valid_df["valido"]).sum())
        }
        out_resumen = output_dir / f"RESUMEN_{nombre}_{nombre_variable}.json"
        with open(out_resumen, "w", encoding="utf-8") as f:
            json.dump(resumen, f, ensure_ascii=False, indent=2)
        print(f"    Guardado resumen JSON: {out_resumen.name}")

        # Metadata de trazabilidad
        guardar_metadata_modelo(
            modelo           = modelo,
            variable         = nombre_variable,
            df_hist          = df_hist,
            out_dir          = output_dir,
            min_muestras_mes = min_muestras_mes,
            sigma_min        = sigma_min
        )

# =====================================================
# 7. RESUMEN FINAL
# =====================================================
print("\n[3/3] ¡Proceso completado!")
print(f"  Todos los archivos están en: {output_dir}")
print(f"  Variables procesadas : {[v['nombre_variable'] for v in VARIABLES_TEMP]}")
print(f"  Modelos procesados   : {len(modelos)}")
archivos_qdm = len(VARIABLES_TEMP) * len(modelos)
print(f"  Archivos QDM totales : {archivos_qdm}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

# =====================================================
# 1. CONFIGURACIÓN DE RUTAS
# =====================================================
# Ruta donde guardaste los archivos de temperatura
drive_base = Path("/content/drive/MyDrive/Project001/salidas_qdm_temperatura_mensual")

modelos = ["MPI-ESM1-2-LR", "EC-Earth3-Veg-LR", "EC-Earth3"]
variables = ["tasmax", "tasmin"]

# Campos que definimos en el script de generación para temperatura
campos_esperados = [
    "mes", "mu_obs_hist", "sigma_obs_hist",
    "mu_mod_hist", "sigma_mod_hist", "n_obs", "n_mod", "estado"
]

def auditar_temperatura_qdm(ruta_csv, modelo, variable):
    sep = "=" * 80
    print(f"\n{sep}")
    print(f"MODELO   : {modelo}")
    print(f"VARIABLE : {variable}")
    print(f"ARCHIVO  : {ruta_csv.name}")
    print(sep)

    if not ruta_csv.exists():
        print(f"[ERROR] Archivo no encontrado.")
        return

    df = pd.read_csv(ruta_csv)
    # Limpiar posibles espacios o BOM
    df.columns = [c.strip().replace("\ufeff", "") for c in df.columns]

    # 1. Verificar integridad de columnas
    faltantes = [c for c in campos_esperados if c not in df.columns]
    if faltantes:
        print(f"[ERROR] Faltan columnas: {faltantes}")
        return

    # 2. Análisis por mes
    print(f"{'Mes':>3} | {'Estado':<12} | {'mu_obs':>7} | {'sigma_obs':>9} | {'mu_mod':>7} | {'sigma_mod':>9} | {'n_obs':>5}")
    print("-" * 80)

    alertas = []
    for _, row in df.iterrows():
        mes = int(row['mes'])
        estado = row['estado']

        # Validaciones lógicas
        if estado == "ok":
            if row['sigma_obs_hist'] <= 0 or row['sigma_mod_hist'] <= 0:
                alertas.append(f"Mes {mes}: Desviación estándar (sigma) no puede ser <= 0.")
            if not np.isfinite(row['mu_obs_hist']) or not np.isfinite(row['mu_mod_hist']):
                alertas.append(f"Mes {mes}: Media (mu) con valores no numéricos (NaN/Inf).")
            if row['n_obs'] < 20:
                alertas.append(f"Mes {mes}: Muestra observada muy pequeña ({row['n_obs']}).")
        else:
            alertas.append(f"Mes {mes}: Estado reportado como '{estado}'.")

        print(f"{mes:3d} | {estado:<12} | {row['mu_obs_hist']:7.2f} | {row['sigma_obs_hist']:9.2f} | "
              f"{row['mu_mod_hist']:7.2f} | {row['sigma_mod_hist']:9.2f} | {int(row['n_obs']):5d}")

    # 3. Resumen final del archivo
    print(f"\nRESUMEN DE AUDITORÍA:")
    if not alertas:
        print("  ✅ TODO CORRECTO: Los 12 meses tienen parámetros válidos para la distribución normal.")
    else:
        print(f"  ⚠️ SE ENCONTRARON {len(alertas)} ADVERTENCIAS:")
        for a in alertas:
            print(f"    - {a}")

# =====================================================
# 2. EJECUCIÓN
# =====================================================
print("INICIANDO AUDITORÍA DE PARÁMETROS QDM (TEMPERATURA)...")

for var in variables:
    for mod in modelos:
        nombre_modelo = mod.replace("-", "_")
        archivo = drive_base / f"QDM_{nombre_modelo}_{var}_monthly_normal.csv"
        auditar_temperatura_qdm(archivo, mod, var)

# Listar archivos JSON de resumen para verificar que existan
print("\n" + "="*80)
print("VERIFICACIÓN DE ARCHIVOS DE APOYO (JSON):")
for var in variables:
    for mod in modelos:
        nombre_modelo = mod.replace("-", "_")
        json_path = drive_base / f"RESUMEN_{nombre_modelo}_{var}.json"
        if json_path.exists():
            with open(json_path, 'r') as f:
                res = json.load(f)
                print(f"  [OK] {json_path.name} | Meses válidos: {res['meses_validos']}/12")
        else:
            print(f"  [FALTA] {json_path.name}")


## 3.5 · Corrección espacial QDM — Precipitación

3 modelos × 2 umbrales (0.1 mm y 0.5 mm), con reinicio seguro y trazabilidad por TXT.


In [ ]:
# ==============================================================
# CORRECCIÓN ESPACIAL QDM MASIVA - 3 MODELOS x 2 UMBRALES
# v5 - Reinicio seguro con TXT + nomenclatura corregido_variable_modelo_umbral_fecha
# ==============================================================

import os
import json
import re
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm  # ← Barra visual nativa de Colab/Jupyter

# --- SILENCIAR ADVERTENCIAS DE RASTERIO/GDAL ---
os.environ['CPL_LOG'] = '/dev/null'

import logging
logging.getLogger('rasterio').setLevel(logging.ERROR)


# =====================================================
# 1. CONFIGURACIÓN
# =====================================================
BASE = Path("/content/drive/MyDrive/Project001")

DIR_TABLAS_QDM  = BASE / "salidas_qdm_mensual"
DIR_ENTRADA_RAW = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_FUTURA_RAW"
DIR_SALIDA_QDM  = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_CORREGIDA_QDM"

DIR_SALIDA_QDM.mkdir(parents=True, exist_ok=True)

UMBRALES_INFO    = {"u0p1": 0.1, "u0p5": 0.5}
MODELOS_OBJETIVO = ["MPI-ESM1-2-LR", "EC-Earth3-Veg-LR", "EC-Earth3"]
#MODELOS_OBJETIVO = ["EC-Earth3"]#

# =====================================================
# 2. UTILIDADES DE CONSOLA
# =====================================================

def ts() -> str:
    return datetime.now().strftime("%H:%M:%S")

def linea(char="─", n=70):
    print(char * n)

def header(texto, char="═"):
    linea(char)
    print(f"  {texto}")
    linea(char)

def ok(texto):
    print(f"  ✔  {texto}")

def warn(texto):
    print(f"  ⚠  {texto}")

def err(texto):
    print(f"  ✘  {texto}")

def inicio_grupo(escenario, tag_u, total, dir_out):
    linea()
    print(f"  ▶ INICIO  [{ts()}]")
    print(f"     Escenario : {escenario}")
    print(f"     Umbral    : {tag_u}")
    print(f"     Archivos  : {total} .tif en cola")
    print(f"     Salida    : {dir_out}")
    linea()

def fin_grupo(escenario, tag_u, procesados, saltados, errores, total):
    estado = "✔ SIN ERRORES" if errores == 0 else f"⚠ {errores} ERROR(ES)"
    linea()
    print(f"  ■ FIN     [{ts()}]  →  {estado}")
    print(f"     Escenario  : {escenario}")
    print(f"     Umbral     : {tag_u}")
    print(f"     Procesados : {procesados}")
    print(f"     Ya existían: {saltados}")
    print(f"     Errores    : {errores}")
    print(f"     Total TIF  : {total}")
    linea()
    print()


# =====================================================
# 3. FUNCIONES PRINCIPALES
# =====================================================

def cargar_biblioteca_qdm() -> dict:
    """Carga las tablas QDM mostrando todos los archivos encontrados."""
    biblioteca = {}
    archivos_csv = sorted(DIR_TABLAS_QDM.glob("QDM_*_monthly.csv"))

    header(f"CARGANDO BIBLIOTECA QDM  —  {len(archivos_csv)} archivo(s) encontrado(s)")

    if not archivos_csv:
        warn("No se encontraron archivos CSV. Verifica DIR_TABLAS_QDM.")
        return biblioteca

    # ── Listar TODOS los archivos detectados ──────────────────────────
    print("  Archivos detectados:")
    for i, f in enumerate(archivos_csv, 1):
        print(f"    [{i:02d}] {f.name}")
    linea("─")

    # ── Procesar cada archivo ─────────────────────────────────────────
    for ruta_csv in tqdm(archivos_csv, desc="  Cargando tablas QDM", unit="csv"):
        tag_u  = next((t for t in UMBRALES_INFO    if t in ruta_csv.name), None)
        modelo = next((m for m in MODELOS_OBJETIVO if m.replace("-", "_") in ruta_csv.name), None)

        if not (tag_u and modelo):
            warn(f"Sin modelo/umbral reconocible: {ruta_csv.name} — omitido.")
            continue

        print(f"\n  ▶ INICIO [{ts()}]  Leyendo: {ruta_csv.name}")
        try:
            df = pd.read_csv(ruta_csv)
            tabla_meses = {}

            for _, row in df[df['estado'] == 'ok'].iterrows():
                probs      = np.array(json.loads(row['probs']))
                q_obs_hist = np.array(json.loads(row['q_obs_hist']))
                q_mod_hist = np.array(json.loads(row['q_mod_hist']))

                # Garantizar orden para np.interp
                if not np.all(np.diff(probs) >= 0):
                    warn(f"  'probs' desordenado en mes {row['mes']} — corrigiendo.")
                    idx = np.argsort(probs)
                    probs, q_obs_hist, q_mod_hist = probs[idx], q_obs_hist[idx], q_mod_hist[idx]

                tabla_meses[int(row['mes'])] = {
                    'probs':      probs,
                    'q_obs_hist': q_obs_hist,
                    'q_mod_hist': q_mod_hist
                }

            biblioteca[(modelo, tag_u)] = tabla_meses
            ok(f"FIN [{ts()}]  →  {len(tabla_meses)} meses cargados  ({modelo}, {tag_u})")

        except Exception as e:
            err(f"Fallo al cargar {ruta_csv.name}: {e}")

    linea("═")
    ok(f"Biblioteca lista: {len(biblioteca)} tabla(s) disponible(s).")
    linea("═")
    print()
    return biblioteca


def aplicar_qdm_espacial(data: np.ndarray, params_mes: dict, valor_umbral: float) -> np.ndarray:
    data_filt = np.where(data < valor_umbral, 0.0, data)
    mask = data_filt > 0
    if not np.any(mask):
        return data_filt

    val_raw  = data_filt[mask]
    p        = np.interp(val_raw, params_mes['q_mod_hist'], params_mes['probs'])
    q_obs_p  = np.interp(p, params_mes['probs'], params_mes['q_obs_hist'])
    q_mod_p  = np.interp(p, params_mes['probs'], params_mes['q_mod_hist'])
    factor   = np.divide(q_obs_p, q_mod_p, out=np.zeros_like(q_obs_p), where=q_mod_p > 1e-9)

    resultado = np.zeros_like(data_filt)
    resultado[mask] = np.round(val_raw * factor, 2)
    return resultado


def extraer_mes(ruta_tif: Path):
    try:
        mes = int(ruta_tif.stem.split('_')[-3])
        return mes if 1 <= mes <= 12 else None
    except (ValueError, IndexError):
        return None


# =====================================================
# 3.1 NOMENCLATURA DE SALIDA
# corregido_variable_modelo_umbral_fecha.tif
# =====================================================

def normalizar_modelo(modelo: str) -> str:
    return modelo.replace("-", "_")

def extraer_variable_desde_nombre(ruta_tif: Path) -> str:
    """
    Toma la primera parte del nombre como variable.
    Ejemplo:
    PR_RAW_MPI_ESM1_2_LR_ssp245_2015_01_WGS84_90m.tif -> PR
    """
    partes = ruta_tif.stem.split("_")
    if len(partes) >= 1:
        return partes[0]
    return "VAR"

def extraer_fecha_desde_nombre(ruta_tif: Path) -> str:
    """
    Busca fechas tipo:
    YYYY_MM_DD  o  YYYY_MM
    y devuelve:
    2015_01_15  o  2015_01
    """
    stem = ruta_tif.stem

    m3 = re.search(r'(\d{4})_(\d{2})_(\d{2})', stem)
    if m3:
        return f"{m3.group(1)}_{m3.group(2)}_{m3.group(3)}"

    m2 = re.search(r'(\d{4})_(\d{2})', stem)
    if m2:
        return f"{m2.group(1)}_{m2.group(2)}"

    return "sin_fecha"

def nombre_salida_qdm(ruta_tif: Path, modelo: str, tag_u: str) -> str:
    """
    Nomenclatura:
    corregido_variable_modelo_umbral_fecha.tif
    """
    variable   = extraer_variable_desde_nombre(ruta_tif)
    modelo_txt = normalizar_modelo(modelo)
    fecha_txt  = extraer_fecha_desde_nombre(ruta_tif)
    return f"corregido_{variable}_{modelo_txt}_{tag_u}_{fecha_txt}.tif"


# =====================================================
# 3.2 REGISTRO LIVIANO EN TXT
# =====================================================

def ruta_registro_txt(dir_out: Path) -> Path:
    return dir_out / "_registro_procesados.txt"

def leer_registro_txt(dir_out: Path) -> list:
    ruta_txt = ruta_registro_txt(dir_out)
    if not ruta_txt.exists():
        return []

    try:
        with open(ruta_txt, "r", encoding="utf-8") as f:
            lineas = [line.strip() for line in f if line.strip()]
        return lineas
    except Exception as e:
        warn(f"No se pudo leer el TXT de registro: {e}")
        return []

def reescribir_registro_txt(dir_out: Path, lista_archivos: list):
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "w", encoding="utf-8") as f:
            for nombre in lista_archivos:
                f.write(f"{nombre}\n")
    except Exception as e:
        warn(f"No se pudo reescribir el TXT de registro: {e}")

def append_registro_txt(dir_out: Path, nombre_archivo: str):
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "a", encoding="utf-8") as f:
            f.write(f"{nombre_archivo}\n")
    except Exception as e:
        warn(f"No se pudo actualizar el TXT de registro: {e}")


# =====================================================
# 3.3 REINICIO SEGURO
# Borra SOLO los 2 últimos .tif corregidos
# =====================================================

def listar_corregidos_por_fecha(dir_out: Path) -> list:
    """
    Lista únicamente archivos corregidos .tif ordenados por fecha de modificación.
    No incluye el TXT ni otros archivos.
    """
    archivos = [
        p for p in dir_out.glob("corregido_*.tif")
        if p.is_file()
    ]
    archivos = sorted(archivos, key=lambda p: p.stat().st_mtime)
    return archivos

def preparar_reinicio_seguro(dir_out: Path):
    """
    Si ya existen archivos corregidos en la carpeta:
    - borra los 2 últimos .tif corregidos creados
    - actualiza el TXT
    """
    archivos_corregidos = listar_corregidos_por_fecha(dir_out)

    if len(archivos_corregidos) == 0:
        return

    ultimos_dos = archivos_corregidos[-2:]
    eliminados = []

    for ruta_arch in ultimos_dos:
        try:
            ruta_arch.unlink()
            eliminados.append(ruta_arch.name)
        except Exception as e:
            warn(f"No se pudo borrar {ruta_arch.name}: {e}")

    if eliminados:
        warn("Reinicio detectado: se borraron los 2 últimos archivos corregidos.")
        for nombre in eliminados:
            print(f"     - Eliminado: {nombre}")

    # reconstruir TXT desde lo que realmente quedó en disco
    restantes = listar_corregidos_por_fecha(dir_out)
    reescribir_registro_txt(dir_out, [p.name for p in restantes])


# =====================================================
# 4. PROCESO PRINCIPAL
# =====================================================

biblioteca = cargar_biblioteca_qdm()

for modelo in MODELOS_OBJETIVO:

    header(f"MODELO: {modelo}", char="█")

    mod_busqueda = modelo.replace("-", "_")
    carpetas_escenarios = [
        d for d in DIR_ENTRADA_RAW.iterdir()
        if d.is_dir() and (modelo in d.name or mod_busqueda in d.name)
    ]

    if not carpetas_escenarios:
        warn(f"No se encontraron carpetas RAW para {modelo}.")
        continue

    for carpeta_in in carpetas_escenarios:

        for tag_u, valor_u in UMBRALES_INFO.items():
            clave = (modelo, tag_u)
            if clave not in biblioteca:
                warn(f"Sin tabla QDM para {clave}. Saltando.")
                continue

            params_qdm   = biblioteca[clave]
            dir_out      = DIR_SALIDA_QDM / f"{carpeta_in.name}_{tag_u}"
            dir_out.mkdir(parents=True, exist_ok=True)

            archivos_tif = sorted(carpeta_in.glob("*.tif"))
            total        = len(archivos_tif)
            errores      = 0
            saltados     = 0
            procesados   = 0
            max_errores  = 5

            # ── Reinicio seguro: borra solo los 2 últimos corregidos ──
            preparar_reinicio_seguro(dir_out)

            inicio_grupo(carpeta_in.name, tag_u, total, dir_out)

            # ── Barra de progreso visual en Colab ─────────────────────
            barra = tqdm(
                archivos_tif,
                total=total,
                desc=f"  {carpeta_in.name} | {tag_u}",
                unit="tif",
                colour="green"
            )

            for ruta_tif in barra:
                ruta_salida = dir_out / nombre_salida_qdm(ruta_tif, modelo, tag_u)

                # Reanudar: saltar si ya existe el corregido con la nueva nomenclatura
                if ruta_salida.exists():
                    saltados += 1
                    barra.set_postfix(proc=procesados, skip=saltados, err=errores)
                    continue

                mes = extraer_mes(ruta_tif)
                if mes is None or mes not in params_qdm:
                    continue

                try:
                    with rasterio.open(ruta_tif) as src:
                        meta = src.meta.copy()
                        meta.update(dtype=rasterio.float32, nodata=0,
                                    photometric="MINISBLACK")

                        with rasterio.open(ruta_salida, 'w', **meta) as dst:
                            for b in range(1, src.count + 1):
                                banda_corr = aplicar_qdm_espacial(
                                    src.read(b), params_qdm[mes], valor_u
                                ).astype(rasterio.float32)
                                dst.write(banda_corr, b)

                    procesados += 1

                    # registrar salida en TXT
                    append_registro_txt(dir_out, ruta_salida.name)

                except Exception as e:
                    errores += 1
                    if errores <= max_errores:
                        err(f"[ERROR {errores}] {ruta_tif.name}: {e}")
                    elif errores == max_errores + 1:
                        warn("Más errores detectados — se silencian.")

                barra.set_postfix(proc=procesados, skip=saltados, err=errores)

            barra.close()

            # reconstrucción final del TXT desde disco para mantener coherencia
            restantes = listar_corregidos_por_fecha(dir_out)
            reescribir_registro_txt(dir_out, [p.name for p in restantes])

            fin_grupo(carpeta_in.name, tag_u, procesados, saltados, errores, total)

header("✔  PROCESO COMPLETO FINALIZADO EXITOSAMENTE", char="█")

## 3.6 · Corrección espacial QDM — Temperatura

Método `x_corr = mu_obs + ((x_raw - mu_mod) / sigma_mod) * sigma_obs`.


In [ ]:
# ==============================================================
# CORRECCIÓN ESPACIAL MENSUAL PARA TEMPERATURA (tasmax / tasmin)
# Usa tablas generadas por:
#   QDM_<modelo>_<variable>_monthly_normal.csv
#
# Lógica:
#   x_corr = mu_obs_hist + ((x_raw - mu_mod_hist) / sigma_mod_hist) * sigma_obs_hist
#
# v2 - Detecta automáticamente rutas reales de entrada
# ==============================================================

import os
import re
import logging
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

# --- SILENCIAR ADVERTENCIAS DE RASTERIO/GDAL ---
os.environ["CPL_LOG"] = "/dev/null"
logging.getLogger("rasterio").setLevel(logging.ERROR)

# =====================================================
# 1. CONFIGURACIÓN GENERAL
# =====================================================

BASE = Path("/content/drive/MyDrive/Project001")

# Tablas generadas por tu script de temperatura
DIR_TABLAS_QDM = BASE / "salidas_qdm_temperatura_mensual"

# ------------------------------------------------------------------
# Ahora se prueban varias rutas posibles para evitar error por nombre
# ------------------------------------------------------------------
VARIABLES_CFG = {
    "tasmax": {
        "dir_entrada_raw_opciones": [
            BASE / "DATOS_CLIMATICOS_TOTALES" / "TEMPERATURA_MAX_FUTURA_RAW",
            BASE / "DATOS_CLIMATICOS_TOTALES" / "TEMPERATURA_MAXIMA_FUTURA_RAW",
        ],
        "dir_salida_corr": BASE / "DATOS_CLIMATICOS_TOTALES" / "TEMPERATURA_MAXIMA_CORREGIDA_QDM",
        "raw_en_kelvin": False,
        "salida_en_kelvin": False
    },
    "tasmin": {
        "dir_entrada_raw_opciones": [
            BASE / "DATOS_CLIMATICOS_TOTALES" / "TEMPERATURA_MIN_FUTURA_RAW",
            BASE / "DATOS_CLIMATICOS_TOTALES" / "TEMPERATURA_MINIMA_FUTURA_RAW",
        ],
        "dir_salida_corr": BASE / "DATOS_CLIMATICOS_TOTALES" / "TEMPERATURA_MINIMA_CORREGIDA_QDM",
        "raw_en_kelvin": False,
        "salida_en_kelvin": False
    }
}

MODELOS_OBJETIVO = [
    "MPI-ESM1-2-LR",
    "EC-Earth3-Veg-LR",
    "EC-Earth3"
]

SIGMA_MIN = 0.01
NODATA_SALIDA_POR_DEFECTO = -9999.0

# =====================================================
# 2. UTILIDADES DE CONSOLA
# =====================================================

def ts() -> str:
    return datetime.now().strftime("%H:%M:%S")

def linea(char="─", n=72):
    print(char * n)

def header(texto, char="═"):
    linea(char)
    print(f"  {texto}")
    linea(char)

def ok(texto):
    print(f"  ✔  {texto}")

def warn(texto):
    print(f"  ⚠  {texto}")

def err(texto):
    print(f"  ✘  {texto}")

def inicio_grupo(variable, escenario, modelo, total, dir_out):
    linea()
    print(f"  ▶ INICIO   [{ts()}]")
    print(f"     Variable  : {variable}")
    print(f"     Escenario : {escenario}")
    print(f"     Modelo    : {modelo}")
    print(f"     Archivos  : {total} .tif en cola")
    print(f"     Salida    : {dir_out}")
    linea()

def fin_grupo(variable, escenario, modelo, procesados, saltados, omitidos, errores, total):
    estado = "✔ SIN ERRORES" if errores == 0 else f"⚠ {errores} ERROR(ES)"
    linea()
    print(f"  ■ FIN      [{ts()}]  →  {estado}")
    print(f"     Variable   : {variable}")
    print(f"     Escenario  : {escenario}")
    print(f"     Modelo     : {modelo}")
    print(f"     Procesados : {procesados}")
    print(f"     Ya existían: {saltados}")
    print(f"     Omitidos   : {omitidos}")
    print(f"     Errores    : {errores}")
    print(f"     Total TIF  : {total}")
    linea()
    print()

# =====================================================
# 3. RESOLUCIÓN DE RUTAS
# =====================================================

def resolver_directorio_existente(opciones):
    """
    Devuelve la primera ruta existente de la lista.
    Si ninguna existe, devuelve la primera como referencia.
    """
    for ruta in opciones:
        if ruta.exists():
            return ruta, True
    return opciones[0], False

# =====================================================
# 4. UTILIDADES DE NOMBRES / PARSEO
# =====================================================

def normalizar_modelo(modelo: str) -> str:
    return modelo.replace("-", "_")

def desnormalizar_modelo(modelo_norm: str) -> str | None:
    for m in MODELOS_OBJETIVO:
        if normalizar_modelo(m) == modelo_norm:
            return m
    return None

def detectar_modelo_y_variable_desde_csv(nombre_csv: str):
    """
    Ejemplo esperado:
      QDM_MPI_ESM1_2_LR_tasmax_monthly_normal.csv
      QDM_EC_Earth3_Veg_LR_tasmin_monthly_normal.csv
    """
    patron = r"^QDM_(.+)_(tasmax|tasmin)_monthly_normal\.csv$"
    m = re.match(patron, nombre_csv)
    if not m:
        return None, None

    modelo_norm = m.group(1)
    variable = m.group(2)
    modelo = desnormalizar_modelo(modelo_norm)
    return modelo, variable

def extraer_variable_desde_nombre(ruta_tif: Path) -> str:
    primer_token = ruta_tif.stem.split("_")[0].lower()

    mapa = {
        "tasmax": "tasmax",
        "tmax":   "tasmax",
        "tx":     "tasmax",
        "tasmin": "tasmin",
        "tmin":   "tasmin",
        "tn":     "tasmin"
    }
    return mapa.get(primer_token, primer_token)

def extraer_fecha_desde_nombre(ruta_tif: Path) -> str:
    stem = ruta_tif.stem

    m3 = re.search(r"(\d{4})_(\d{2})_(\d{2})", stem)
    if m3:
        return f"{m3.group(1)}_{m3.group(2)}_{m3.group(3)}"

    m2 = re.search(r"(\d{4})_(\d{2})", stem)
    if m2:
        return f"{m2.group(1)}_{m2.group(2)}"

    return "sin_fecha"

def extraer_mes(ruta_tif: Path):
    stem = ruta_tif.stem

    m3 = re.search(r"(\d{4})_(\d{2})_(\d{2})", stem)
    if m3:
        mes = int(m3.group(2))
        return mes if 1 <= mes <= 12 else None

    m2 = re.search(r"(\d{4})_(\d{2})", stem)
    if m2:
        mes = int(m2.group(2))
        return mes if 1 <= mes <= 12 else None

    return None

def nombre_salida_temp(ruta_tif: Path, modelo: str, variable: str) -> str:
    modelo_txt = normalizar_modelo(modelo)
    fecha_txt  = extraer_fecha_desde_nombre(ruta_tif)
    return f"corregido_{variable}_{modelo_txt}_{fecha_txt}.tif"

# =====================================================
# 5. REGISTRO LIVIANO EN TXT
# =====================================================

def ruta_registro_txt(dir_out: Path) -> Path:
    return dir_out / "_registro_procesados.txt"

def leer_registro_txt(dir_out: Path) -> list:
    ruta_txt = ruta_registro_txt(dir_out)
    if not ruta_txt.exists():
        return []

    try:
        with open(ruta_txt, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]
    except Exception as e:
        warn(f"No se pudo leer el TXT de registro: {e}")
        return []

def reescribir_registro_txt(dir_out: Path, lista_archivos: list):
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "w", encoding="utf-8") as f:
            for nombre in lista_archivos:
                f.write(f"{nombre}\n")
    except Exception as e:
        warn(f"No se pudo reescribir el TXT de registro: {e}")

def append_registro_txt(dir_out: Path, nombre_archivo: str):
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "a", encoding="utf-8") as f:
            f.write(f"{nombre_archivo}\n")
    except Exception as e:
        warn(f"No se pudo actualizar el TXT de registro: {e}")

# =====================================================
# 6. REINICIO SEGURO
# =====================================================

def listar_corregidos_por_fecha(dir_out: Path) -> list:
    archivos = [
        p for p in dir_out.glob("corregido_*.tif")
        if p.is_file()
    ]
    return sorted(archivos, key=lambda p: p.stat().st_mtime)

def preparar_reinicio_seguro(dir_out: Path):
    archivos_corregidos = listar_corregidos_por_fecha(dir_out)

    if len(archivos_corregidos) == 0:
        return

    ultimos_dos = archivos_corregidos[-2:]
    eliminados = []

    for ruta_arch in ultimos_dos:
        try:
            ruta_arch.unlink()
            eliminados.append(ruta_arch.name)
        except Exception as e:
            warn(f"No se pudo borrar {ruta_arch.name}: {e}")

    if eliminados:
        warn("Reinicio detectado: se borraron los 2 últimos archivos corregidos.")
        for nombre in eliminados:
            print(f"     - Eliminado: {nombre}")

    restantes = listar_corregidos_por_fecha(dir_out)
    reescribir_registro_txt(dir_out, [p.name for p in restantes])

# =====================================================
# 7. CARGA DE BIBLIOTECA QDM DE TEMPERATURA
# =====================================================

def cargar_biblioteca_qdm_temperatura() -> dict:
    biblioteca = {}
    archivos_csv = sorted(DIR_TABLAS_QDM.glob("QDM_*_monthly_normal.csv"))

    header(f"CARGANDO BIBLIOTECA QDM-TEMP  —  {len(archivos_csv)} archivo(s) encontrado(s)")

    if not archivos_csv:
        warn("No se encontraron archivos CSV de temperatura.")
        warn(f"Verifica esta ruta: {DIR_TABLAS_QDM}")
        return biblioteca

    print("  Archivos detectados:")
    for i, f in enumerate(archivos_csv, 1):
        print(f"    [{i:02d}] {f.name}")
    linea("─")

    for ruta_csv in tqdm(archivos_csv, desc="  Cargando tablas T°", unit="csv"):
        modelo, variable = detectar_modelo_y_variable_desde_csv(ruta_csv.name)

        if not (modelo and variable):
            warn(f"Nombre no reconocible: {ruta_csv.name} — omitido.")
            continue

        print(f"\n  ▶ INICIO [{ts()}]  Leyendo: {ruta_csv.name}")

        try:
            df = pd.read_csv(ruta_csv)

            columnas_necesarias = {
                "mes", "mu_obs_hist", "sigma_obs_hist",
                "mu_mod_hist", "sigma_mod_hist", "estado"
            }
            faltantes = columnas_necesarias - set(df.columns)
            if faltantes:
                warn(f"Faltan columnas en {ruta_csv.name}: {sorted(faltantes)}")
                continue

            tabla_meses = {}

            for _, row in df[df["estado"] == "ok"].iterrows():
                mes = int(row["mes"])

                tabla_meses[mes] = {
                    "mu_obs_hist":    float(row["mu_obs_hist"]),
                    "sigma_obs_hist": max(float(row["sigma_obs_hist"]), SIGMA_MIN),
                    "mu_mod_hist":    float(row["mu_mod_hist"]),
                    "sigma_mod_hist": max(float(row["sigma_mod_hist"]), SIGMA_MIN)
                }

            biblioteca[(modelo, variable)] = tabla_meses
            ok(f"FIN [{ts()}]  →  {len(tabla_meses)} meses cargados  ({modelo}, {variable})")

        except Exception as e:
            err(f"Fallo al cargar {ruta_csv.name}: {e}")

    linea("═")
    ok(f"Biblioteca lista: {len(biblioteca)} tabla(s) disponible(s).")
    linea("═")
    print()

    return biblioteca

# =====================================================
# 8. CORRECCIÓN ESPACIAL DE TEMPERATURA
# =====================================================

def resolver_nodata_salida(src_nodata):
    if src_nodata is None:
        return float(NODATA_SALIDA_POR_DEFECTO)

    try:
        val = float(src_nodata)
        if np.isfinite(val):
            return val
    except Exception:
        pass

    return float(NODATA_SALIDA_POR_DEFECTO)

def aplicar_correccion_temperatura_normal(
    data: np.ndarray,
    params_mes: dict,
    nodata_in=None,
    raw_en_kelvin=False,
    salida_en_kelvin=False
) -> np.ndarray:
    arr = np.asarray(data, dtype=np.float32)

    mask_valida = np.isfinite(arr)

    if nodata_in is not None:
        try:
            nodata_in = float(nodata_in)
            if np.isfinite(nodata_in):
                mask_valida &= (arr != nodata_in)
        except Exception:
            pass

    if not np.any(mask_valida):
        return arr.astype(np.float32), mask_valida

    x = arr[mask_valida].astype(np.float32)

    if raw_en_kelvin:
        x = x - 273.15

    mu_obs    = float(params_mes["mu_obs_hist"])
    sigma_obs = max(float(params_mes["sigma_obs_hist"]), SIGMA_MIN)
    mu_mod    = float(params_mes["mu_mod_hist"])
    sigma_mod = max(float(params_mes["sigma_mod_hist"]), SIGMA_MIN)

    z = (x - mu_mod) / sigma_mod
    x_corr = mu_obs + z * sigma_obs

    if salida_en_kelvin:
        x_corr = x_corr + 273.15

    x_corr = np.round(x_corr, 2)

    return x_corr.astype(np.float32), mask_valida

# =====================================================
# 9. BÚSQUEDA DE ESCENARIOS
# =====================================================

def carpetas_modelo_en_directorio(dir_base: Path, modelo: str) -> list:
    if not dir_base.exists():
        return []

    modelo_norm = normalizar_modelo(modelo)

    carpetas = [
        d for d in dir_base.iterdir()
        if d.is_dir() and (modelo in d.name or modelo_norm in d.name)
    ]

    return sorted(carpetas)

def archivos_tif_directos_modelo(dir_base: Path, modelo: str) -> list:
    if not dir_base.exists():
        return []

    modelo_norm = normalizar_modelo(modelo)

    tifs = [
        f for f in dir_base.glob("*.tif")
        if (modelo in f.name or modelo_norm in f.name)
    ]

    return sorted(tifs)

# =====================================================
# 10. PROCESO PRINCIPAL
# =====================================================

biblioteca = cargar_biblioteca_qdm_temperatura()

if not biblioteca:
    raise RuntimeError("No se pudo construir la biblioteca QDM de temperatura.")

for variable, cfg in VARIABLES_CFG.items():

    dir_entrada, existe_dir = resolver_directorio_existente(cfg["dir_entrada_raw_opciones"])
    dir_salida_base = cfg["dir_salida_corr"]
    raw_en_kelvin = cfg["raw_en_kelvin"]
    salida_en_kelvin = cfg["salida_en_kelvin"]

    dir_salida_base.mkdir(parents=True, exist_ok=True)

    header(f"VARIABLE: {variable}", char="█")
    print(f"  Entrada RAW : {dir_entrada}")
    print(f"  Salida QDM  : {dir_salida_base}")
    print(f"  RAW en K    : {raw_en_kelvin}")
    print(f"  Salida en K : {salida_en_kelvin}")
    print()

    if not existe_dir:
        warn(f"No existe el directorio de entrada para {variable}.")
        print("  Se buscaron estas rutas:")
        for ruta in cfg["dir_entrada_raw_opciones"]:
            print(f"     - {ruta}")
        print()
        continue
    else:
        ok(f"Directorio detectado correctamente para {variable}: {dir_entrada}")
        print()

    for modelo in MODELOS_OBJETIVO:
        clave = (modelo, variable)

        if clave not in biblioteca:
            warn(f"Sin tabla QDM para {clave}. Saltando.")
            continue

        params_qdm = biblioteca[clave]

        # 10.1 Buscar carpetas de escenarios por modelo
        carpetas_escenarios = carpetas_modelo_en_directorio(dir_entrada, modelo)

        # ---------------------------------------------------------
        # CASO A: hay subcarpetas por escenario
        # ---------------------------------------------------------
        if carpetas_escenarios:
            for carpeta_in in carpetas_escenarios:
                dir_out = dir_salida_base / carpeta_in.name
                dir_out.mkdir(parents=True, exist_ok=True)

                archivos_tif = sorted(carpeta_in.glob("*.tif"))
                total = len(archivos_tif)

                procesados = 0
                saltados   = 0
                omitidos   = 0
                errores    = 0
                max_errores = 5

                preparar_reinicio_seguro(dir_out)
                inicio_grupo(variable, carpeta_in.name, modelo, total, dir_out)

                barra = tqdm(
                    archivos_tif,
                    total=total,
                    desc=f"  {variable} | {carpeta_in.name}",
                    unit="tif",
                    colour="green"
                )

                for ruta_tif in barra:
                    ruta_salida = dir_out / nombre_salida_temp(ruta_tif, modelo, variable)

                    if ruta_salida.exists():
                        saltados += 1
                        barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)
                        continue

                    mes = extraer_mes(ruta_tif)
                    if mes is None or mes not in params_qdm:
                        omitidos += 1
                        barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)
                        continue

                    try:
                        with rasterio.open(ruta_tif) as src:
                            src_nodata = src.nodata
                            nodata_out = resolver_nodata_salida(src_nodata)

                            meta = src.meta.copy()
                            meta.update(
                                dtype=rasterio.float32,
                                nodata=nodata_out,
                                compress="lzw"
                            )

                            with rasterio.open(ruta_salida, "w", **meta) as dst:
                                for b in range(1, src.count + 1):
                                    data = src.read(b)

                                    banda_corr, mask_valida = aplicar_correccion_temperatura_normal(
                                        data=data,
                                        params_mes=params_qdm[mes],
                                        nodata_in=src_nodata,
                                        raw_en_kelvin=raw_en_kelvin,
                                        salida_en_kelvin=salida_en_kelvin
                                    )

                                    out_band = np.full(
                                        data.shape,
                                        nodata_out,
                                        dtype=np.float32
                                    )
                                    out_band[mask_valida] = banda_corr

                                    dst.write(out_band, b)

                        procesados += 1
                        append_registro_txt(dir_out, ruta_salida.name)

                    except Exception as e:
                        errores += 1
                        if errores <= max_errores:
                            err(f"[ERROR {errores}] {ruta_tif.name}: {e}")
                        elif errores == max_errores + 1:
                            warn("Más errores detectados — se silencian.")

                    barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)

                barra.close()

                restantes = listar_corregidos_por_fecha(dir_out)
                reescribir_registro_txt(dir_out, [p.name for p in restantes])

                fin_grupo(variable, carpeta_in.name, modelo, procesados, saltados, omitidos, errores, total)

        # ---------------------------------------------------------
        # CASO B: no hay subcarpetas, pero sí TIFF directos
        # ---------------------------------------------------------
        else:
            archivos_tif = archivos_tif_directos_modelo(dir_entrada, modelo)

            if not archivos_tif:
                warn(f"No se encontraron carpetas ni TIFF directos para {modelo} en {dir_entrada}")
                continue

            nombre_grupo = f"{normalizar_modelo(modelo)}_directo"
            dir_out = dir_salida_base / nombre_grupo
            dir_out.mkdir(parents=True, exist_ok=True)

            total = len(archivos_tif)
            procesados = 0
            saltados   = 0
            omitidos   = 0
            errores    = 0
            max_errores = 5

            preparar_reinicio_seguro(dir_out)
            inicio_grupo(variable, nombre_grupo, modelo, total, dir_out)

            barra = tqdm(
                archivos_tif,
                total=total,
                desc=f"  {variable} | {nombre_grupo}",
                unit="tif",
                colour="green"
            )

            for ruta_tif in barra:
                ruta_salida = dir_out / nombre_salida_temp(ruta_tif, modelo, variable)

                if ruta_salida.exists():
                    saltados += 1
                    barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)
                    continue

                mes = extraer_mes(ruta_tif)
                if mes is None or mes not in params_qdm:
                    omitidos += 1
                    barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)
                    continue

                try:
                    with rasterio.open(ruta_tif) as src:
                        src_nodata = src.nodata
                        nodata_out = resolver_nodata_salida(src_nodata)

                        meta = src.meta.copy()
                        meta.update(
                            dtype=rasterio.float32,
                            nodata=nodata_out,
                            compress="lzw"
                        )

                        with rasterio.open(ruta_salida, "w", **meta) as dst:
                            for b in range(1, src.count + 1):
                                data = src.read(b)

                                banda_corr, mask_valida = aplicar_correccion_temperatura_normal(
                                    data=data,
                                    params_mes=params_qdm[mes],
                                    nodata_in=src_nodata,
                                    raw_en_kelvin=raw_en_kelvin,
                                    salida_en_kelvin=salida_en_kelvin
                                )

                                out_band = np.full(
                                    data.shape,
                                    nodata_out,
                                    dtype=np.float32
                                )
                                out_band[mask_valida] = banda_corr

                                dst.write(out_band, b)

                    procesados += 1
                    append_registro_txt(dir_out, ruta_salida.name)

                except Exception as e:
                    errores += 1
                    if errores <= max_errores:
                        err(f"[ERROR {errores}] {ruta_tif.name}: {e}")
                    elif errores == max_errores + 1:
                        warn("Más errores detectados — se silencian.")

                barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)

            barra.close()

            restantes = listar_corregidos_por_fecha(dir_out)
            reescribir_registro_txt(dir_out, [p.name for p in restantes])

            fin_grupo(variable, nombre_grupo, modelo, procesados, saltados, omitidos, errores, total)

header("✔ PROCESO COMPLETO FINALIZADO EXITOSAMENTE", char="█")

## 3.7 · Detección de faltantes y reparación


In [ ]:
# ==============================================================
# DETECTAR FALTANTES CORREGIDOS Y REDESCARGAR RAW
# PARA TODOS LOS MODELOS, ESCENARIOS Y VARIABLES
# ==============================================================
# Este script:
# 1) Recorre todos los MODELOS y SCENARIOS
# 2) Recorre tasmax y tasmin
# 3) Detecta qué corregido_*.tif faltan
# 4) Verifica si el RAW correspondiente existe
# 5) Si no existe o está dañado, lo descarga desde GEE
# 6) Genera un reporte TXT por cada combinación
#
# NOTA:
# - Este script recupera los RAW faltantes.
# - NO rehace el archivo corregido QDM, porque para eso se necesita
#   tu script de corrección QDM.
# ==============================================================

import os
import math
import time
from pathlib import Path

import ee
import geemap
from tqdm.notebook import tqdm
from google.colab import drive

# ==============================================================
# 1. MONTAR GOOGLE DRIVE
# ==============================================================

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# ==============================================================
# 2. CONFIGURACIÓN GENERAL
# ==============================================================

PROJECT_ID = "computer-492420"
BASE = Path("/content/drive/MyDrive/Project001")
DIR_CLIMA_TOTAL = BASE / "DATOS_CLIMATICOS_TOTALES"

AOI_ASSET = "projects/mapas2025-473512/assets/AreaAporte"
PIXEL_M = 90.0

ANIO_INI = 2015
ANIO_FIN = 2100

TIMEOUT_DESCARGA = 1200
MAX_REINTENTOS = 3
MIN_BYTES_VALIDO = 10 * 1024  # 10 KB

# ==============================================================
# 3. MODELOS Y ESCENARIOS
# ==============================================================
# Estos son los mismos de tu script original

MODELOS = [
    "MPI-ESM1-2-LR",
    "EC-Earth3-Veg-LR",
    "EC-Earth3",
]

SCENARIOS = [
    "ssp245",
    "ssp585",
]

# ==============================================================
# 4. MAPEO DE NOMBRES PARA SALIDA CORREGIDA
# ==============================================================
# Aquí defines cómo aparece el nombre del modelo dentro del archivo corregido.
#
# Ejemplo que tú mostraste:
# carpeta       : EC_Earth3_Veg_LR_ssp245
# archivo QDM   : corregido_tasmin_EC_Earth3_2017_06.tif
#
# Por eso EC-Earth3-Veg-LR -> EC_Earth3 en "modelo_salida"
#
# Ajusta este diccionario si tus nombres corregidos reales son distintos.

MODELO_SALIDA_MAP = {
    "MPI-ESM1-2-LR": "MPI_ESM1_2_LR",
    "EC-Earth3-Veg-LR": "EC_Earth3",
    "EC-Earth3": "EC_Earth3",
}

# ==============================================================
# 5. FUNCIONES AUXILIARES
# ==============================================================

def inicializar_gee():
    try:
        ee.Initialize(project=PROJECT_ID)
        print("✅ Earth Engine inicializado.")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID)
        print("✅ Earth Engine autenticado e inicializado.")


def construir_grilla_exportacion():
    aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
    bbox = aoi_geom.bounds()

    ring = ee.List(ee.List(bbox.coordinates().get(0)))
    xmin = ee.Number(ee.List(ring.get(0)).get(0))
    ymax = ee.Number(ee.List(ring.get(2)).get(1))

    lat = ee.Number(bbox.centroid(1).coordinates().get(1))

    deg_y = ee.Number(PIXEL_M / 111320.0)
    deg_x = ee.Number(PIXEL_M).divide(
        ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos())
    )

    xform = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])
    return bbox, xform


def archivo_valido(path_obj):
    return path_obj.exists() and path_obj.stat().st_size >= MIN_BYTES_VALIDO


def nombre_corregido(variable, modelo_salida, anio, mes):
    return f"corregido_{variable}_{modelo_salida}_{anio}_{mes:02d}.tif"


def nombre_id_mes(modelo_carpeta, escenario, anio, mes):
    return f"{modelo_carpeta}_{escenario}_{anio}_{mes:02d}_WGS84_90m"


def nombre_raw(prefijo_raw, id_mes):
    return f"{prefijo_raw}_RAW_{id_mes}.tif"


def construir_caso(variable, modelo_gee, escenario):
    modelo_carpeta = modelo_gee.replace("-", "_")
    modelo_salida = MODELO_SALIDA_MAP.get(modelo_gee, modelo_carpeta)

    if variable == "tasmin":
        dir_corregida = DIR_CLIMA_TOTAL / "TEMPERATURA_MINIMA_CORREGIDA_QDM" / f"{modelo_carpeta}_{escenario}"
        dir_raw = DIR_CLIMA_TOTAL / "TEMPERATURA_MIN_FUTURA_RAW" / f"{modelo_carpeta}_{escenario}"
        prefijo_raw = "TMIN"
    elif variable == "tasmax":
        dir_corregida = DIR_CLIMA_TOTAL / "TEMPERATURA_MAXIMA_CORREGIDA_QDM" / f"{modelo_carpeta}_{escenario}"
        dir_raw = DIR_CLIMA_TOTAL / "TEMPERATURA_MAX_FUTURA_RAW" / f"{modelo_carpeta}_{escenario}"
        prefijo_raw = "TMAX"
    else:
        raise ValueError(f"Variable no soportada: {variable}")

    return {
        "variable": variable,
        "escenario": escenario,
        "modelo_gee": modelo_gee,
        "modelo_carpeta": modelo_carpeta,
        "modelo_salida": modelo_salida,
        "dir_corregida": dir_corregida,
        "dir_raw": dir_raw,
        "prefijo_raw": prefijo_raw,
    }


def detectar_faltantes_corregidos(caso):
    carpeta = caso["dir_corregida"]
    carpeta.mkdir(parents=True, exist_ok=True)

    existentes = {p.name for p in carpeta.glob("*.tif")}
    faltantes = []

    for anio in range(ANIO_INI, ANIO_FIN + 1):
        for mes in range(1, 13):
            esperado = nombre_corregido(
                caso["variable"],
                caso["modelo_salida"],
                anio,
                mes
            )
            if esperado not in existentes:
                faltantes.append({
                    "anio": anio,
                    "mes": mes,
                    "archivo_corregido": esperado
                })

    return existentes, faltantes


def descargar_raw_mes(caso, anio, mes, bbox, xform):
    variable = caso["variable"]
    escenario = caso["escenario"]
    modelo_gee = caso["modelo_gee"]
    modelo_carpeta = caso["modelo_carpeta"]
    prefijo_raw = caso["prefijo_raw"]
    dir_raw = caso["dir_raw"]

    dir_raw.mkdir(parents=True, exist_ok=True)

    id_mes = nombre_id_mes(modelo_carpeta, escenario, anio, mes)
    raw_path = dir_raw / nombre_raw(prefijo_raw, id_mes)

    if archivo_valido(raw_path):
        return {
            "status": "raw_ya_existia",
            "raw_path": str(raw_path),
            "anio": anio,
            "mes": mes
        }

    t_inicio = ee.Date.fromYMD(anio, mes, 1)

    col = (
        ee.ImageCollection("NASA/GDDP-CMIP6")
        .filter(ee.Filter.eq("model", modelo_gee))
        .filter(ee.Filter.eq("scenario", escenario))
        .filterDate(t_inicio, t_inicio.advance(1, "month"))
        .select(variable)
    )

    n_imgs = int(col.size().getInfo())
    if n_imgs == 0:
        raise RuntimeError(
            f"No hay imágenes para {variable} | {modelo_gee} | {escenario} | {anio}-{mes:02d}"
        )

    col_celsius = col.map(
        lambda img: img.subtract(273.15).copyProperties(img, ["system:time_start"])
    )

    fechas = ee.List(col_celsius.aggregate_array("system:time_start")).map(
        lambda t: ee.Date(t).format("YYYYMMdd")
    )

    img_export = (
        col_celsius.toBands()
        .rename(fechas)
        .clip(bbox)
        .reproject(crs="EPSG:4326", crsTransform=xform)
        .toFloat()
    )

    geemap.ee_export_image(
        img_export,
        filename=str(raw_path),
        region=bbox,
        crs="EPSG:4326",
        crs_transform=xform,
        timeout=TIMEOUT_DESCARGA
    )

    if not archivo_valido(raw_path):
        raise IOError(f"El archivo descargado quedó vacío o incompleto: {raw_path}")

    return {
        "status": "raw_descargado",
        "raw_path": str(raw_path),
        "anio": anio,
        "mes": mes
    }


def procesar_caso(caso, bbox, xform):
    print("\n" + "=" * 80)
    print(f"Variable       : {caso['variable']}")
    print(f"Modelo GEE     : {caso['modelo_gee']}")
    print(f"Modelo carpeta : {caso['modelo_carpeta']}")
    print(f"Modelo salida  : {caso['modelo_salida']}")
    print(f"Escenario      : {caso['escenario']}")
    print(f"Dir corregida  : {caso['dir_corregida']}")
    print(f"Dir raw        : {caso['dir_raw']}")
    print("=" * 80)

    existentes, faltantes = detectar_faltantes_corregidos(caso)

    total_esperados = (ANIO_FIN - ANIO_INI + 1) * 12

    print(f"Esperados : {total_esperados}")
    print(f"Existentes: {len(existentes)}")
    print(f"Faltan    : {len(faltantes)}")

    if not faltantes:
        print("✅ No faltan corregidos en esta combinación.")
        return

    reporte = caso["dir_corregida"] / f"_reporte_recuperacion_{caso['variable']}_{caso['modelo_carpeta']}_{caso['escenario']}.txt"

    descargados = 0
    ya_existian = 0
    fallidos = 0

    with open(reporte, "w", encoding="utf-8") as rep:
        rep.write("REPORTE DE RECUPERACIÓN\n")
        rep.write("=" * 80 + "\n")
        rep.write(f"Variable       : {caso['variable']}\n")
        rep.write(f"Modelo GEE     : {caso['modelo_gee']}\n")
        rep.write(f"Modelo carpeta : {caso['modelo_carpeta']}\n")
        rep.write(f"Modelo salida  : {caso['modelo_salida']}\n")
        rep.write(f"Escenario      : {caso['escenario']}\n")
        rep.write(f"Dir corregida  : {caso['dir_corregida']}\n")
        rep.write(f"Dir raw        : {caso['dir_raw']}\n")
        rep.write("=" * 80 + "\n\n")

        for item in faltantes:
            rep.write(f"FALTANTE: {item['archivo_corregido']}\n")

        rep.write("\n" + "-" * 80 + "\n\n")

        pbar = tqdm(faltantes, desc=f"{caso['variable']} | {caso['modelo_carpeta']} | {caso['escenario']}", unit="mes")

        for item in pbar:
            anio = item["anio"]
            mes = item["mes"]
            archivo_corregido = item["archivo_corregido"]
            pbar.set_postfix(fecha=f"{anio}-{mes:02d}")

            ok = False
            ultimo_error = None

            for intento in range(1, MAX_REINTENTOS + 1):
                try:
                    resultado = descargar_raw_mes(caso, anio, mes, bbox, xform)

                    rep.write(
                        f"[OK] {archivo_corregido} | {anio}-{mes:02d} | {resultado['status']} | {resultado['raw_path']}\n"
                    )

                    if resultado["status"] == "raw_descargado":
                        descargados += 1
                    else:
                        ya_existian += 1

                    ok = True
                    break

                except Exception as e:
                    ultimo_error = str(e)
                    if intento < MAX_REINTENTOS:
                        time.sleep(3)

            if not ok:
                fallidos += 1
                rep.write(
                    f"[ERROR] {archivo_corregido} | {anio}-{mes:02d} | {ultimo_error}\n"
                )
                print(f"❌ {archivo_corregido} -> {ultimo_error}")

        rep.write("\n" + "=" * 80 + "\n")
        rep.write(f"RAW descargados : {descargados}\n")
        rep.write(f"RAW ya existían : {ya_existian}\n")
        rep.write(f"Fallidos        : {fallidos}\n")

    print("\nResumen:")
    print(f"RAW descargados : {descargados}")
    print(f"RAW ya existían : {ya_existian}")
    print(f"Fallidos        : {fallidos}")
    print(f"Reporte         : {reporte}")


# ==============================================================
# 6. GENERAR TODOS LOS CASOS
# ==============================================================

CASOS = []

for modelo in MODELOS:
    for escenario in SCENARIOS:
        for variable in ["tasmax", "tasmin"]:
            CASOS.append(construir_caso(variable, modelo, escenario))

print(f"Total de casos a revisar: {len(CASOS)}")

# ==============================================================
# 7. EJECUCIÓN
# ==============================================================

inicializar_gee()
bbox, xform = construir_grilla_exportacion()

for caso in CASOS:
    caso["dir_corregida"].mkdir(parents=True, exist_ok=True)
    caso["dir_raw"].mkdir(parents=True, exist_ok=True)
    procesar_caso(caso, bbox, xform)

print("\n✅ PROCESO COMPLETADO PARA TODOS LOS MODELOS Y ESCENARIOS.")

In [ ]:
# ==============================================================
# PRECIPITACIÓN QDM MASIVA
# DETECCIÓN DE FALTANTES + DESCARGA RAW SI FALTA + GENERACIÓN
#
# Estructura real de salida corregida:
#   PRECIPITACION_CORREGIDA_QDM/<modelo>_<escenario>_<umbral>/
#   corregido_PR_<modelo>_<umbral>_<YYYY_MM>.tif
#
# Estructura real de RAW:
#   PRECIPITACION_FUTURA_RAW/<modelo>_<escenario>/
#   PR_RAW_<modelo>_<escenario>_<YYYY_MM>_WGS84_90m.tif
#
# Tablas QDM:
#   salidas_qdm_mensual/QDM_*_monthly.csv
#
# Usa la misma lógica de tu script real de precipitación:
# - 2 umbrales: u0p1, u0p5
# - QDM espacial con probs, q_obs_hist, q_mod_hist
# - reinicio seguro opcional
# ==============================================================

import os
import json
import re
import math
import time
import logging
import contextlib
import numpy as np
import pandas as pd
import rasterio
import ee
import geemap

from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

# --------------------------------------------------------------
# SILENCIAR ADVERTENCIAS
# --------------------------------------------------------------
os.environ["CPL_LOG"] = "/dev/null"
logging.getLogger("rasterio").setLevel(logging.ERROR)

# =====================================================
# 1. CONFIGURACIÓN GENERAL
# =====================================================

BASE = Path("/content/drive/MyDrive/Project001")

DIR_TABLAS_QDM  = BASE / "salidas_qdm_mensual"
DIR_ENTRADA_RAW = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_FUTURA_RAW"
DIR_SALIDA_QDM  = BASE / "DATOS_CLIMATICOS_TOTALES" / "PRECIPITACION_CORREGIDA_QDM"

DIR_ENTRADA_RAW.mkdir(parents=True, exist_ok=True)
DIR_SALIDA_QDM.mkdir(parents=True, exist_ok=True)

AOI_ASSET = "projects/mapas2025-473512/assets/AreaAporte"
PROJECT_ID = "computer-492420"
PIXEL_M = 90.0

ANIO_INI = 2015
ANIO_FIN = 2100

MODELOS_OBJETIVO = [
    "MPI-ESM1-2-LR",
    "EC-Earth3-Veg-LR",
    "EC-Earth3"
]

SCENARIOS = ["ssp245", "ssp585"]
UMBRALES_INFO = {"u0p1": 0.1, "u0p5": 0.5}

MIN_BYTES_VALIDO = 10 * 1024
TIMEOUT_DESCARGA = 1200
MAX_REINTENTOS_DESCARGA = 3

ACTIVAR_REINICIO_SEGURO = False
N_BORRAR_REINICIO = 2

# =====================================================
# 2. UTILIDADES DE CONSOLA
# =====================================================

def ts() -> str:
    return datetime.now().strftime("%H:%M:%S")

def linea(char="─", n=78):
    print(char * n)

def header(texto, char="═"):
    linea(char)
    print(f"  {texto}")
    linea(char)

def ok(texto):
    print(f"  ✔  {texto}")

def warn(texto):
    print(f"  ⚠  {texto}")

def err(texto):
    print(f"  ✘  {texto}")

def inicio_grupo(modelo, escenario, tag_u, total_esperados, existentes, faltan, dir_out):
    linea()
    print(f"  ▶ INICIO   [{ts()}]")
    print(f"     Modelo     : {modelo}")
    print(f"     Escenario  : {escenario}")
    print(f"     Umbral     : {tag_u}")
    print(f"     Esperados  : {total_esperados}")
    print(f"     Existentes : {existentes}")
    print(f"     Faltan     : {faltan}")
    print(f"     Salida     : {dir_out}")
    linea()

def fin_grupo(modelo, escenario, tag_u, generados, raw_descargados, ya_existian, omitidos, errores, total):
    estado = "✔ SIN ERRORES" if errores == 0 else f"⚠ {errores} ERROR(ES)"
    linea()
    print(f"  ■ FIN      [{ts()}]  →  {estado}")
    print(f"     Modelo         : {modelo}")
    print(f"     Escenario      : {escenario}")
    print(f"     Umbral         : {tag_u}")
    print(f"     Generados      : {generados}")
    print(f"     RAW descargados: {raw_descargados}")
    print(f"     Ya existían    : {ya_existian}")
    print(f"     Omitidos       : {omitidos}")
    print(f"     Errores        : {errores}")
    print(f"     Total esperado : {total}")
    linea()
    print()

# =====================================================
# 3. UTILIDADES DE NOMBRES
# =====================================================

def normalizar_modelo(modelo: str) -> str:
    return modelo.replace("-", "_")

def archivo_valido(path_obj: Path) -> bool:
    return path_obj.exists() and path_obj.is_file() and path_obj.stat().st_size >= MIN_BYTES_VALIDO

def nombre_raw_pr(modelo: str, escenario: str, anio: int, mes: int) -> str:
    modelo_txt = normalizar_modelo(modelo)
    return f"PR_RAW_{modelo_txt}_{escenario}_{anio}_{mes:02d}_WGS84_90m.tif"

def nombre_corregido_pr(modelo: str, tag_u: str, anio: int, mes: int) -> str:
    modelo_txt = normalizar_modelo(modelo)
    return f"corregido_PR_{modelo_txt}_{tag_u}_{anio}_{mes:02d}.tif"

def extraer_mes_desde_raw(ruta_tif: Path):
    try:
        mes = int(ruta_tif.stem.split("_")[-3])
        return mes if 1 <= mes <= 12 else None
    except (ValueError, IndexError):
        return None

# =====================================================
# 4. REGISTRO TXT
# =====================================================

def ruta_registro_txt(dir_out: Path) -> Path:
    return dir_out / "_registro_procesados.txt"

def reescribir_registro_txt(dir_out: Path):
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        archivos = sorted([p.name for p in dir_out.glob("corregido_*.tif") if p.is_file()])
        with open(ruta_txt, "w", encoding="utf-8") as f:
            for nombre in archivos:
                f.write(f"{nombre}\n")
    except Exception as e:
        warn(f"No se pudo reescribir el TXT de registro: {e}")

def append_registro_txt(dir_out: Path, nombre_archivo: str):
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "a", encoding="utf-8") as f:
            f.write(f"{nombre_archivo}\n")
    except Exception as e:
        warn(f"No se pudo actualizar el TXT de registro: {e}")

# =====================================================
# 5. REINICIO SEGURO
# =====================================================

def listar_corregidos_por_fecha(dir_out: Path) -> list:
    archivos = [p for p in dir_out.glob("corregido_*.tif") if p.is_file()]
    return sorted(archivos, key=lambda p: p.stat().st_mtime)

def preparar_reinicio_seguro(dir_out: Path):
    if not ACTIVAR_REINICIO_SEGURO:
        return

    archivos_corregidos = listar_corregidos_por_fecha(dir_out)
    if len(archivos_corregidos) == 0:
        return

    ultimos = archivos_corregidos[-N_BORRAR_REINICIO:]
    eliminados = []

    for ruta_arch in ultimos:
        try:
            ruta_arch.unlink()
            eliminados.append(ruta_arch.name)
        except Exception as e:
            warn(f"No se pudo borrar {ruta_arch.name}: {e}")

    if eliminados:
        warn(f"Reinicio detectado: se borraron los últimos {len(eliminados)} corregidos.")
        for nombre in eliminados:
            print(f"     - Eliminado: {nombre}")

    reescribir_registro_txt(dir_out)

# =====================================================
# 6. CARGA DE BIBLIOTECA QDM
# =====================================================

def cargar_biblioteca_qdm() -> dict:
    """
    Carga tablas QDM de precipitación con patrón:
      QDM_*_monthly.csv
    y genera biblioteca[(modelo, tag_u)][mes] = {...}
    """
    biblioteca = {}
    archivos_csv = sorted(DIR_TABLAS_QDM.glob("QDM_*_monthly.csv"))

    header(f"CARGANDO BIBLIOTECA QDM  —  {len(archivos_csv)} archivo(s) encontrado(s)")

    if not archivos_csv:
        warn("No se encontraron archivos CSV. Verifica DIR_TABLAS_QDM.")
        return biblioteca

    print("  Archivos detectados:")
    for i, f in enumerate(archivos_csv, 1):
        print(f"    [{i:02d}] {f.name}")
    linea("─")

    for ruta_csv in tqdm(archivos_csv, desc="  Cargando tablas QDM", unit="csv"):
        tag_u  = next((t for t in UMBRALES_INFO if t in ruta_csv.name), None)
        modelo = next((m for m in MODELOS_OBJETIVO if m.replace("-", "_") in ruta_csv.name), None)

        if not (tag_u and modelo):
            warn(f"Sin modelo/umbral reconocible: {ruta_csv.name} — omitido.")
            continue

        print(f"\n  ▶ INICIO [{ts()}]  Leyendo: {ruta_csv.name}")

        try:
            df = pd.read_csv(ruta_csv)
            tabla_meses = {}

            for _, row in df[df["estado"] == "ok"].iterrows():
                probs      = np.array(json.loads(row["probs"]))
                q_obs_hist = np.array(json.loads(row["q_obs_hist"]))
                q_mod_hist = np.array(json.loads(row["q_mod_hist"]))

                if not np.all(np.diff(probs) >= 0):
                    warn(f"'probs' desordenado en mes {row['mes']} — corrigiendo.")
                    idx = np.argsort(probs)
                    probs, q_obs_hist, q_mod_hist = probs[idx], q_obs_hist[idx], q_mod_hist[idx]

                tabla_meses[int(row["mes"])] = {
                    "probs": probs,
                    "q_obs_hist": q_obs_hist,
                    "q_mod_hist": q_mod_hist
                }

            biblioteca[(modelo, tag_u)] = tabla_meses
            ok(f"FIN [{ts()}]  →  {len(tabla_meses)} meses cargados ({modelo}, {tag_u})")

        except Exception as e:
            err(f"Fallo al cargar {ruta_csv.name}: {e}")

    linea("═")
    ok(f"Biblioteca lista: {len(biblioteca)} tabla(s) disponible(s).")
    linea("═")
    print()

    return biblioteca

# =====================================================
# 7. GEE Y GRILLA
# =====================================================

def inicializar_gee():
    try:
        ee.Initialize(project=PROJECT_ID)
        ok("Earth Engine inicializado.")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID)
        ok("Earth Engine autenticado e inicializado.")

def construir_grilla_exportacion():
    aoi_geom = ee.FeatureCollection(AOI_ASSET).geometry()
    bbox = aoi_geom.bounds()

    ring = ee.List(ee.List(bbox.coordinates().get(0)))
    xmin = ee.Number(ee.List(ring.get(0)).get(0))
    ymax = ee.Number(ee.List(ring.get(2)).get(1))
    lat = ee.Number(bbox.centroid(1).coordinates().get(1))

    deg_y = ee.Number(PIXEL_M / 111320.0)
    deg_x = ee.Number(PIXEL_M).divide(
        ee.Number(111320.0).multiply(lat.multiply(math.pi / 180).cos())
    )

    xform = ee.List([deg_x, 0, xmin, 0, deg_y.multiply(-1), ymax])
    return bbox, xform

# =====================================================
# 8. DESCARGA RAW SI FALTA
# =====================================================

def descargar_raw_mes_pr(modelo: str, escenario: str, anio: int, mes: int, ruta_raw: Path, bbox, xform):
    """
    Descarga el RAW de precipitación y lo guarda en mm/día,
    igual que tu descargador original.
    """
    t_inicio = ee.Date.fromYMD(anio, mes, 1)

    col = (
        ee.ImageCollection("NASA/GDDP-CMIP6")
        .filter(ee.Filter.eq("model", modelo))
        .filter(ee.Filter.eq("scenario", escenario))
        .filterDate(t_inicio, t_inicio.advance(1, "month"))
        .select("pr")
    )

    n_imgs = int(col.size().getInfo())
    if n_imgs == 0:
        raise RuntimeError(f"No hay imágenes en GEE para pr | {modelo} | {escenario} | {anio}-{mes:02d}")

    col_mm = col.map(
        lambda img: img.multiply(86400.0).copyProperties(img, ["system:time_start"])
    )

    fechas = ee.List(col_mm.aggregate_array("system:time_start")).map(
        lambda t: ee.Date(t).format("YYYYMMdd")
    )

    img_export = (
        col_mm.toBands()
        .rename(fechas)
        .clip(bbox)
        .reproject(crs="EPSG:4326", crsTransform=xform)
        .toFloat()
    )

    ruta_raw.parent.mkdir(parents=True, exist_ok=True)

    with open(os.devnull, "w") as fnull:
        with contextlib.redirect_stdout(fnull):
            geemap.ee_export_image(
                img_export,
                filename=str(ruta_raw),
                region=bbox,
                crs="EPSG:4326",
                crs_transform=xform,
                timeout=TIMEOUT_DESCARGA
            )

    if not archivo_valido(ruta_raw):
        raise IOError(f"El RAW descargado quedó vacío o incompleto: {ruta_raw}")

# =====================================================
# 9. CORRECCIÓN QDM ESPACIAL
# =====================================================

def aplicar_qdm_espacial(data: np.ndarray, params_mes: dict, valor_umbral: float) -> np.ndarray:
    """
    Igual que tu script real:
    - pone a cero los valores menores al umbral
    - calcula percentil p usando q_mod_hist
    - obtiene q_obs_p y q_mod_p
    - aplica factor multiplicativo q_obs_p / q_mod_p
    """
    data_filt = np.where(data < valor_umbral, 0.0, data)
    mask = data_filt > 0

    if not np.any(mask):
        return data_filt.astype(np.float32)

    val_raw = data_filt[mask]

    p = np.interp(val_raw, params_mes["q_mod_hist"], params_mes["probs"])
    q_obs_p = np.interp(p, params_mes["probs"], params_mes["q_obs_hist"])
    q_mod_p = np.interp(p, params_mes["probs"], params_mes["q_mod_hist"])

    factor = np.divide(
        q_obs_p,
        q_mod_p,
        out=np.zeros_like(q_obs_p),
        where=q_mod_p > 1e-9
    )

    resultado = np.zeros_like(data_filt, dtype=np.float32)
    resultado[mask] = np.round(val_raw * factor, 2)
    return resultado.astype(np.float32)

def corregir_raw_pr_a_qdm(ruta_raw: Path, ruta_salida: Path, params_mes: dict, valor_umbral: float):
    ruta_salida.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(ruta_raw) as src:
        meta = src.meta.copy()
        meta.update(
            dtype=rasterio.float32,
            nodata=0,
            photometric="MINISBLACK"
        )

        with rasterio.open(ruta_salida, "w", **meta) as dst:
            for b in range(1, src.count + 1):
                data = src.read(b)
                banda_corr = aplicar_qdm_espacial(
                    data=data,
                    params_mes=params_mes,
                    valor_umbral=valor_umbral
                ).astype(rasterio.float32)

                dst.write(banda_corr, b)

    if not archivo_valido(ruta_salida):
        raise IOError(f"El corregido generado quedó vacío o incompleto: {ruta_salida}")

# =====================================================
# 10. DETECCIÓN DE FALTANTES
# =====================================================

def detectar_faltantes_corregidos_pr(dir_out: Path, modelo: str, tag_u: str):
    """
    Detecta faltantes según la nomenclatura real:
      corregido_PR_<modelo_norm>_<tag_u>_<YYYY_MM>.tif
    """
    faltantes = []
    existentes = 0

    for anio in range(ANIO_INI, ANIO_FIN + 1):
        for mes in range(1, 13):
            nombre = nombre_corregido_pr(modelo, tag_u, anio, mes)
            ruta = dir_out / nombre

            if ruta.exists():
                existentes += 1
            else:
                faltantes.append({
                    "anio": anio,
                    "mes": mes,
                    "nombre_salida": nombre,
                    "ruta_salida": ruta
                })

    total = (ANIO_FIN - ANIO_INI + 1) * 12
    return total, existentes, faltantes

# =====================================================
# 11. PROCESO PRINCIPAL POR COMBINACIÓN
# =====================================================

def procesar_combinacion_pr(modelo: str, escenario: str, tag_u: str, valor_u: float, params_qdm: dict, bbox, xform):
    modelo_norm = normalizar_modelo(modelo)

    dir_in = DIR_ENTRADA_RAW / f"{modelo_norm}_{escenario}"
    dir_out = DIR_SALIDA_QDM / f"{modelo_norm}_{escenario}_{tag_u}"

    dir_in.mkdir(parents=True, exist_ok=True)
    dir_out.mkdir(parents=True, exist_ok=True)

    preparar_reinicio_seguro(dir_out)

    total_esperados, existentes, faltantes = detectar_faltantes_corregidos_pr(
        dir_out=dir_out,
        modelo=modelo,
        tag_u=tag_u
    )

    inicio_grupo(modelo, escenario, tag_u, total_esperados, existentes, len(faltantes), dir_out)

    if not faltantes:
        fin_grupo(modelo, escenario, tag_u, generados=0, raw_descargados=0, ya_existian=existentes, omitidos=0, errores=0, total=total_esperados)
        return

    generados = 0
    raw_descargados = 0
    ya_existian = existentes
    omitidos = 0
    errores = 0
    max_errores_visibles = 5

    reporte = dir_out / f"_reporte_faltantes_PR_{modelo_norm}_{escenario}_{tag_u}.txt"

    with open(reporte, "w", encoding="utf-8") as rep:
        rep.write("REPORTE DE DETECCIÓN Y GENERACIÓN DE FALTANTES - PRECIPITACIÓN\n")
        rep.write("=" * 80 + "\n")
        rep.write(f"Modelo        : {modelo}\n")
        rep.write(f"Escenario     : {escenario}\n")
        rep.write(f"Umbral        : {tag_u}\n")
        rep.write(f"Dir RAW       : {dir_in}\n")
        rep.write(f"Dir salida    : {dir_out}\n")
        rep.write("=" * 80 + "\n\n")
        rep.write(f"Esperados  : {total_esperados}\n")
        rep.write(f"Existentes : {existentes}\n")
        rep.write(f"Faltantes  : {len(faltantes)}\n\n")

        barra = tqdm(
            faltantes,
            total=len(faltantes),
            desc=f"  PR | {modelo_norm}_{escenario}_{tag_u}",
            unit="mes",
            colour="green"
        )

        for item in barra:
            anio = item["anio"]
            mes = item["mes"]
            ruta_salida = item["ruta_salida"]
            nombre_sal = item["nombre_salida"]

            barra.set_postfix(fecha=f"{anio}-{mes:02d}", gen=generados, raw=raw_descargados, omi=omitidos, err=errores)

            if ruta_salida.exists():
                ya_existian += 1
                continue

            if mes not in params_qdm:
                omitidos += 1
                rep.write(f"[OMITIDO] {nombre_sal} -> No hay parámetros QDM para mes={mes}\n")
                continue

            ruta_raw = dir_in / nombre_raw_pr(modelo, escenario, anio, mes)

            try:
                # 1) Descargar RAW si falta
                if not archivo_valido(ruta_raw):
                    ok_descarga = False
                    ultimo_error_descarga = None

                    for intento in range(1, MAX_REINTENTOS_DESCARGA + 1):
                        try:
                            descargar_raw_mes_pr(
                                modelo=modelo,
                                escenario=escenario,
                                anio=anio,
                                mes=mes,
                                ruta_raw=ruta_raw,
                                bbox=bbox,
                                xform=xform
                            )
                            raw_descargados += 1
                            ok_descarga = True
                            break
                        except Exception as e:
                            ultimo_error_descarga = str(e)
                            if intento < MAX_REINTENTOS_DESCARGA:
                                time.sleep(3)

                    if not ok_descarga:
                        raise RuntimeError(f"No se pudo descargar RAW: {ultimo_error_descarga}")

                # 2) Generar corregido
                corregir_raw_pr_a_qdm(
                    ruta_raw=ruta_raw,
                    ruta_salida=ruta_salida,
                    params_mes=params_qdm[mes],
                    valor_umbral=valor_u
                )

                generados += 1
                append_registro_txt(dir_out, ruta_salida.name)
                rep.write(f"[OK] {nombre_sal} | generado desde {ruta_raw.name}\n")

            except Exception as e:
                errores += 1
                rep.write(f"[ERROR] {nombre_sal} -> {e}\n")

                if errores <= max_errores_visibles:
                    err(f"[ERROR {errores}] {nombre_sal}: {e}")
                elif errores == max_errores_visibles + 1:
                    warn("Más errores detectados — se silencian.")

            barra.set_postfix(fecha=f"{anio}-{mes:02d}", gen=generados, raw=raw_descargados, omi=omitidos, err=errores)

        barra.close()

        rep.write("\n" + "=" * 80 + "\n")
        rep.write(f"Generados      : {generados}\n")
        rep.write(f"RAW descargados: {raw_descargados}\n")
        rep.write(f"Ya existían    : {ya_existian}\n")
        rep.write(f"Omitidos       : {omitidos}\n")
        rep.write(f"Errores        : {errores}\n")

    reescribir_registro_txt(dir_out)

    fin_grupo(
        modelo=modelo,
        escenario=escenario,
        tag_u=tag_u,
        generados=generados,
        raw_descargados=raw_descargados,
        ya_existian=ya_existian,
        omitidos=omitidos,
        errores=errores,
        total=total_esperados
    )

# =====================================================
# 12. EJECUCIÓN GENERAL
# =====================================================

header("CONFIGURACIÓN GENERAL", char="█")
print(f"  DIR_TABLAS_QDM  : {DIR_TABLAS_QDM}")
print(f"  DIR_ENTRADA_RAW : {DIR_ENTRADA_RAW}")
print(f"  DIR_SALIDA_QDM  : {DIR_SALIDA_QDM}")
print()

if not DIR_TABLAS_QDM.exists():
    raise RuntimeError(f"No existe DIR_TABLAS_QDM: {DIR_TABLAS_QDM}")

biblioteca = cargar_biblioteca_qdm()
if not biblioteca:
    raise RuntimeError("No se pudo construir la biblioteca QDM de precipitación.")

inicializar_gee()
bbox, xform = construir_grilla_exportacion()

for modelo in MODELOS_OBJETIVO:
    for escenario in SCENARIOS:
        for tag_u, valor_u in UMBRALES_INFO.items():
            clave = (modelo, tag_u)

            if clave not in biblioteca:
                warn(f"Sin tabla QDM para {clave}. Saltando.")
                continue

            procesar_combinacion_pr(
                modelo=modelo,
                escenario=escenario,
                tag_u=tag_u,
                valor_u=valor_u,
                params_qdm=biblioteca[clave],
                bbox=bbox,
                xform=xform
            )

header("✔ PROCESO COMPLETO DE PRECIPITACIÓN FINALIZADO", char="█")

## 3.8 · Utilidades — inventario y árbol de carpetas


In [ ]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/Project001")

def tam_humano(nbytes):
    unidades = ["B", "KB", "MB", "GB", "TB"]
    size = float(nbytes)
    for u in unidades:
        if size < 1024 or u == unidades[-1]:
            return f"{size:.2f} {u}"
        size /= 1024

def calcular_tamano_carpeta(ruta: Path):
    total = 0
    for root, dirs, files in os.walk(ruta):
        for f in files:
            fp = Path(root) / f
            try:
                total += fp.stat().st_size
            except:
                pass
    return total

carpetas = [p for p in BASE.iterdir() if p.is_dir()]
ranking = []

for c in carpetas:
    ranking.append((c.name, calcular_tamano_carpeta(c), c))

ranking = sorted(ranking, key=lambda x: x[1], reverse=True)

print("=" * 70)
print("CARPETAS MÁS PESADAS DE PROJECT001")
print("=" * 70)

for i, (nombre, size, ruta) in enumerate(ranking, 1):
    print(f"{i:02d}. {nombre}")
    print(f"    Ruta   : {ruta}")
    print(f"    Tamaño : {tam_humano(size)}")

In [ ]:
from pathlib import Path

# =========================================================
# 1. RUTA BASE
# =========================================================
BASE = Path("/content/drive/MyDrive/Project001")

# =========================================================
# 2. PARÁMETROS
# =========================================================
MAX_NIVEL = 5
MAX_ELEMENTOS_POR_CARPETA = 6

# =========================================================
# 3. FUNCIÓN PARA IMPRIMIR ÁRBOL LIMITADO
# =========================================================
def imprimir_arbol_limitado(ruta: Path, nivel=0, max_nivel=5, max_elementos=2):
    if nivel > max_nivel or not ruta.exists() or not ruta.is_dir():
        return

    try:
        elementos = sorted(ruta.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    except Exception as e:
        print("    " * nivel + f"└── [Sin acceso] {ruta.name} -> {e}")
        return

    elementos_mostrados = elementos[:max_elementos]

    for elem in elementos_mostrados:
        prefijo = "    " * nivel + "└── "
        if elem.is_dir():
            print(f"{prefijo}📁 {elem.name}")
            imprimir_arbol_limitado(elem, nivel + 1, max_nivel, max_elementos)
        else:
            print(f"{prefijo}📄 {elem.name}")

    if len(elementos) > max_elementos:
        restantes = len(elementos) - max_elementos
        print("    " * nivel + f"└── ... ({restantes} elemento(s) más)")

# =========================================================
# 4. EJECUCIÓN
# =========================================================
if not BASE.exists():
    print(f"No existe la ruta: {BASE}")
else:
    print(f"ESTRUCTURA DE: {BASE}")
    print("=" * 80)
    print(f"📁 {BASE.name}")
    imprimir_arbol_limitado(
        BASE,
        nivel=1,
        max_nivel=MAX_NIVEL,
        max_elementos=MAX_ELEMENTOS_POR_CARPETA
    )

In [ ]:
from pathlib import Path

# =========================================================
# 1. RUTA BASE
# =========================================================
BASE = Path("/content/drive/MyDrive/Project001")

# =========================================================
# 2. PARÁMETROS
# =========================================================
MAX_NIVEL = 5
MAX_TIF_POR_CARPETA = 2

# =========================================================
# 3. FUNCIÓN PARA IMPRIMIR ÁRBOL
# =========================================================
def imprimir_arbol_filtrado(ruta: Path, nivel=0, max_nivel=5, max_tif=2):
    if nivel > max_nivel or not ruta.exists() or not ruta.is_dir():
        return

    try:
        elementos = sorted(ruta.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    except Exception as e:
        print("    " * nivel + f"└── [Sin acceso] {ruta.name} -> {e}")
        return

    carpetas = [e for e in elementos if e.is_dir()]
    archivos_no_tif = [e for e in elementos if e.is_file() and e.suffix.lower() != ".tif"]
    archivos_tif = [e for e in elementos if e.is_file() and e.suffix.lower() == ".tif"]

    # Mostrar TODAS las carpetas
    for carpeta in carpetas:
        prefijo = "    " * nivel + "└── "
        print(f"{prefijo}📁 {carpeta.name}")
        imprimir_arbol_filtrado(carpeta, nivel + 1, max_nivel, max_tif)

    # Mostrar TODOS los archivos que no son tif
    for archivo in archivos_no_tif:
        prefijo = "    " * nivel + "└── "
        print(f"{prefijo}📄 {archivo.name}")

    # Mostrar solo algunos tif
    tif_mostrados = archivos_tif[:max_tif]
    for archivo in tif_mostrados:
        prefijo = "    " * nivel + "└── "
        print(f"{prefijo}🖼️ {archivo.name}")

    if len(archivos_tif) > max_tif:
        restantes = len(archivos_tif) - max_tif
        print("    " * nivel + f"└── ... ({restantes} archivo(s) .tif más)")

# =========================================================
# 4. EJECUCIÓN
# =========================================================
if not BASE.exists():
    print(f"No existe la ruta: {BASE}")
else:
    print(f"ESTRUCTURA DE: {BASE}")
    print("=" * 80)
    print(f"📁 {BASE.name}")
    imprimir_arbol_filtrado(BASE, nivel=1, max_nivel=MAX_NIVEL, max_tif=MAX_TIF_POR_CARPETA)

## 3.9 · ET Hargreaves desde temperatura corregida QDM

Genera ET diaria mensual (bandas = días) por modelo y escenario, a partir de tasmax/tasmin corregidos.


In [ ]:
# ==========================================================
# ET HARGREAVES DESDE TEMPERATURA CORREGIDA QDM
# Lee tasmax / tasmin corregidas y genera ET diaria mensual
# (bandas = días) por modelo y escenario
# ==========================================================

import os
import re
import math
import calendar
import numpy as np
import rasterio
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

# ==========================================================
# 1) CONFIGURACIÓN GENERAL
# ==========================================================
RUTA_BASE = Path("/content/drive/MyDrive/Project001")

DIR_CLIMA_TOTAL = RUTA_BASE / "DATOS_CLIMATICOS_TOTALES"

# Candidatas para temperatura corregida
CANDIDATAS_TMAX = [
    DIR_CLIMA_TOTAL / "TEMPERATURA_MAXIMA_CORREGIDA_QDM",
    DIR_CLIMA_TOTAL / "TEMPERATURA_MAX_CORREGIDA_QDM",
]

CANDIDATAS_TMIN = [
    DIR_CLIMA_TOTAL / "TEMPERATURA_MINIMA_CORREGIDA_QDM",
    DIR_CLIMA_TOTAL / "TEMPERATURA_MIN_CORREGIDA_QDM",
]

# Carpeta de salida para ET corregida
CARPETA_ET_CORREGIDA = DIR_CLIMA_TOTAL / "ET_HARGREAVES_CORREGIDA_QDM"
CARPETA_ET_CORREGIDA.mkdir(parents=True, exist_ok=True)

# Temperatura corregida en Celsius
ENTRADA_EN_CELSIUS = True

# Si ya existe el archivo, no lo recalcula
SOBREESCRIBIR = False

# Nodata de salida
NODATA_OUT = -9999.0

# ==========================================================
# 2) UTILIDADES
# ==========================================================
def primera_ruta_existente(lista_rutas):
    for ruta in lista_rutas:
        if ruta.exists():
            return ruta
    return None

DIR_TMAX = primera_ruta_existente(CANDIDATAS_TMAX)
DIR_TMIN = primera_ruta_existente(CANDIDATAS_TMIN)

if DIR_TMAX is None:
    raise FileNotFoundError(
        "No se encontró carpeta de tasmax corregida. Revisar:\n" +
        "\n".join(str(p) for p in CANDIDATAS_TMAX)
    )

if DIR_TMIN is None:
    raise FileNotFoundError(
        "No se encontró carpeta de tasmin corregida. Revisar:\n" +
        "\n".join(str(p) for p in CANDIDATAS_TMIN)
    )

print("TMAX corregida:", DIR_TMAX)
print("TMIN corregida:", DIR_TMIN)
print("ET corregida  :", CARPETA_ET_CORREGIDA)

def existe_archivo(path):
    return path.exists() and (not SOBREESCRIBIR)

def extraer_anio_mes(nombre_archivo):
    """
    Busca patrones tipo:
      ..._2015_01.tif
      ..._2015_01_algo.tif
    """
    m = re.search(r'(\d{4})_(\d{2})(?:\D|$)', nombre_archivo)
    if m:
        anio = int(m.group(1))
        mes = int(m.group(2))
        if 1 <= mes <= 12:
            return anio, mes
    return None, None

def listar_tifs_por_anio_mes(carpeta):
    """
    Devuelve diccionario {(anio, mes): Path}
    """
    salida = {}
    for p in sorted(carpeta.glob("*.tif")):
        anio, mes = extraer_anio_mes(p.name)
        if anio is not None:
            salida[(anio, mes)] = p
    return salida

def escenarios_comunes(dir_a, dir_b):
    a = {p.name: p for p in dir_a.iterdir() if p.is_dir()}
    b = {p.name: p for p in dir_b.iterdir() if p.is_dir()}
    comunes = sorted(set(a) & set(b))
    return [(a[n], b[n]) for n in comunes]

def fechas_desde_descripciones_o_mes(src, anio, mes):
    """
    Intenta usar descripciones de bandas YYYYMMDD.
    Si no existen, las infiere secuencialmente según el mes.
    """
    descs = list(src.descriptions) if src.descriptions else []

    fechas = []
    if len(descs) == src.count and all(d is not None and str(d).strip() != "" for d in descs):
        ok = True
        for d in descs:
            try:
                fechas.append(datetime.strptime(str(d), "%Y%m%d"))
            except Exception:
                ok = False
                break
        if ok:
            return fechas

    n_dias_mes = calendar.monthrange(anio, mes)[1]
    if src.count > n_dias_mes:
        raise ValueError(
            f"El archivo tiene {src.count} bandas, pero {anio}-{mes:02d} solo tiene {n_dias_mes} días."
        )

    fechas = [datetime(anio, mes, d) for d in range(1, src.count + 1)]
    return fechas

def lat_rad_por_fila(transform, height):
    """
    Calcula latitud del centro de cada fila en radianes.
    Para raster norte-arriba en EPSG:4326, la latitud depende de la fila.
    """
    filas = np.arange(height, dtype=np.float64)
    lat_deg = transform.f + transform.e * (filas + 0.5)
    return np.deg2rad(lat_deg).reshape(-1, 1)

def ra_hargreaves_por_fila(lat_rad_2d, doy):
    """
    Replica la misma lógica del script original:
      delta = 0.409 * sin(2*pi*doy/365 - 1.39)
      omega_s = acos(clamp(-tan(phi)*tan(delta), -1, 1))
      ra = 37.6 * [ ... ]
    """
    delta = 0.409 * np.sin((2.0 * math.pi * doy / 365.0) - 1.39)
    cos_omega_s = -np.tan(lat_rad_2d) * math.tan(delta)
    cos_omega_s = np.clip(cos_omega_s, -1.0, 1.0)
    omega_s = np.arccos(cos_omega_s)

    ra = 37.6 * (
        omega_s * np.sin(lat_rad_2d) * math.sin(delta)
        + np.cos(lat_rad_2d) * math.cos(delta) * np.sin(omega_s)
    )
    return ra.astype(np.float32)

def calcular_et_hargreaves_numpy(tmax, tmin, ra_2d, nodata_mask):
    """
    Replica la fórmula del código original:
      t_mean = (tmax + tmin)/2
      t_range = abs(tmax - tmin)
      et = ra * 0.0009384 * (t_mean + 17.8) * sqrt(t_range)
    """
    t_mean = (tmax + tmin) / 2.0
    t_range = np.abs(tmax - tmin)

    et = ra_2d * 0.0009384 * (t_mean + 17.8) * np.sqrt(t_range)
    et = et.astype(np.float32)

    salida = np.full(tmax.shape, NODATA_OUT, dtype=np.float32)
    salida[~nodata_mask] = et[~nodata_mask]
    return salida

# ==========================================================
# 3) PROCESAMIENTO DE ET DESDE TEMPERATURA CORREGIDA
# ==========================================================
def procesar_escenario(dir_tmax_esc, dir_tmin_esc, dir_out_esc):
    dir_out_esc.mkdir(parents=True, exist_ok=True)

    tmax_files = listar_tifs_por_anio_mes(dir_tmax_esc)
    tmin_files = listar_tifs_por_anio_mes(dir_tmin_esc)

    claves = sorted(set(tmax_files.keys()) & set(tmin_files.keys()))

    if not claves:
        print(f"[WARN] Sin pares mensuales comunes en: {dir_tmax_esc.name}")
        return

    barra = tqdm(claves, desc=f"ET {dir_tmax_esc.name}", unit="mes")

    for (anio, mes) in barra:
        nombre_out = f"ET_HARGREAVES_DAILY_{dir_tmax_esc.name}_{anio}_{mes:02d}_WGS84_GRID90m.tif"
        out_path = dir_out_esc / nombre_out

        if existe_archivo(out_path):
            barra.set_postfix(estado="SKIP", fecha=f"{anio}-{mes:02d}")
            continue

        ruta_tmax = tmax_files[(anio, mes)]
        ruta_tmin = tmin_files[(anio, mes)]

        try:
            with rasterio.open(ruta_tmax) as src_max, rasterio.open(ruta_tmin) as src_min:
                if src_max.count != src_min.count:
                    raise ValueError(
                        f"Número de bandas distinto: tmax={src_max.count}, tmin={src_min.count}"
                    )

                if src_max.width != src_min.width or src_max.height != src_min.height:
                    raise ValueError("Dimensiones distintas entre tasmax y tasmin.")

                if src_max.transform != src_min.transform:
                    raise ValueError("Transform distinta entre tasmax y tasmin.")

                meta = src_max.meta.copy()
                meta.update(
                    dtype=rasterio.float32,
                    count=src_max.count,
                    nodata=NODATA_OUT,
                    compress="lzw"
                )

                fechas = fechas_desde_descripciones_o_mes(src_max, anio, mes)
                lat_rad_2d = lat_rad_por_fila(src_max.transform, src_max.height)

                nodata_max = src_max.nodata
                nodata_min = src_min.nodata

                with rasterio.open(out_path, "w", **meta) as dst:
                    for b in range(1, src_max.count + 1):
                        fecha = fechas[b - 1]
                        doy = fecha.timetuple().tm_yday
                        ra_2d = ra_hargreaves_por_fila(lat_rad_2d, doy)

                        tmax = src_max.read(b).astype(np.float32)
                        tmin = src_min.read(b).astype(np.float32)

                        # Si las entradas estuvieran en Kelvin, convertir aquí
                        if not ENTRADA_EN_CELSIUS:
                            tmax = tmax - 273.15
                            tmin = tmin - 273.15

                        nodata_mask = ~np.isfinite(tmax) | ~np.isfinite(tmin)

                        if nodata_max is not None:
                            nodata_mask |= (tmax == nodata_max)
                        if nodata_min is not None:
                            nodata_mask |= (tmin == nodata_min)

                        et = calcular_et_hargreaves_numpy(
                            tmax=tmax,
                            tmin=tmin,
                            ra_2d=ra_2d,
                            nodata_mask=nodata_mask
                        )

                        dst.write(et, b)
                        dst.set_band_description(b, fecha.strftime("%Y%m%d"))

                barra.set_postfix(estado="OK", fecha=f"{anio}-{mes:02d}")

        except Exception as e:
            barra.set_postfix(estado="ERROR", fecha=f"{anio}-{mes:02d}")
            print(f"[ERROR] {dir_tmax_esc.name} | {anio}-{mes:02d}: {e}")

# ==========================================================
# 4) EJECUCIÓN
# ==========================================================
print("\n" + "=" * 70)
print("ET HARGREAVES DESDE TEMPERATURA CORREGIDA")
print("=" * 70)

pares_escenarios = escenarios_comunes(DIR_TMAX, DIR_TMIN)

if not pares_escenarios:
    raise RuntimeError("No se encontraron carpetas comunes de escenario entre tasmax y tasmin.")

for dir_tmax_esc, dir_tmin_esc in pares_escenarios:
    nombre_escenario = dir_tmax_esc.name
    dir_out_esc = CARPETA_ET_CORREGIDA / nombre_escenario

    print(f"\nProcesando escenario/modelo: {nombre_escenario}")
    print(f"  TMAX: {dir_tmax_esc}")
    print(f"  TMIN: {dir_tmin_esc}")
    print(f"  OUT : {dir_out_esc}")

    procesar_escenario(dir_tmax_esc, dir_tmin_esc, dir_out_esc)

print("\n" + "=" * 70)
print("PROCESO COMPLETADO")
print("=" * 70)
print("TMAX corregida:", DIR_TMAX)
print("TMIN corregida:", DIR_TMIN)
print("ET corregida  :", CARPETA_ET_CORREGIDA)

In [ ]:
# ==========================================================
# ET HARGREAVES DESDE TEMPERATURA CORREGIDA QDM
# ----------------------------------------------------------
# ¿Qué hace este script?
# ----------------------------------------------------------
# 1. Busca las carpetas de temperatura máxima y mínima corregidas.
# 2. Detecta los escenarios/modelos comunes entre tasmax y tasmin.
# 3. Lee cada TIFF mensual corregido:
#       - un TIFF mensual
#       - varias bandas diarias
# 4. Calcula ET diaria con Hargreaves-Samani para cada banda.
# 5. Guarda un TIFF mensual de ET con bandas diarias.
# 6. Muestra una barra de avance.
# 7. Si el proceso se interrumpe y luego se reinicia:
#       - borra los 2 últimos TIFF ET creados en esa carpeta
#       - reconstruye el TXT
#       - recalcula esos 2 archivos
#
# Este diseño ayuda a evitar archivos corruptos si el proceso se
# cortó mientras estaba escribiendo uno de los últimos resultados.
# ==========================================================

import os
import re
import math
import calendar
import logging
import numpy as np
import rasterio

from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

# ----------------------------------------------------------
# Silenciar mensajes muy verbosos de rasterio/GDAL
# ----------------------------------------------------------
os.environ["CPL_LOG"] = "/dev/null"
logging.getLogger("rasterio").setLevel(logging.ERROR)


# ==========================================================
# 1) CONFIGURACIÓN GENERAL
# ==========================================================
# Ruta principal del proyecto en Drive
BASE = Path("/content/drive/MyDrive/Project001")

# Carpeta general donde están todos los datos climáticos
DIR_CLIMA_TOTAL = BASE / "DATOS_CLIMATICOS_TOTALES"

# ----------------------------------------------------------
# Posibles nombres de carpeta para temperatura máxima corregida
# Se prueban varias rutas porque en tu proyecto a veces cambian
# ligeramente los nombres.
# ----------------------------------------------------------
CANDIDATAS_TMAX = [
    DIR_CLIMA_TOTAL / "TEMPERATURA_MAXIMA_CORREGIDA_QDM",
    DIR_CLIMA_TOTAL / "TEMPERATURA_MAX_CORREGIDA_QDM",
]

# ----------------------------------------------------------
# Posibles nombres de carpeta para temperatura mínima corregida
# ----------------------------------------------------------
CANDIDATAS_TMIN = [
    DIR_CLIMA_TOTAL / "TEMPERATURA_MINIMA_CORREGIDA_QDM",
    DIR_CLIMA_TOTAL / "TEMPERATURA_MIN_CORREGIDA_QDM",
]

# ----------------------------------------------------------
# Carpeta principal donde se guardará la ET calculada
# a partir de la temperatura corregida.
# Dentro de esta carpeta se crearán subcarpetas por escenario/modelo.
# ----------------------------------------------------------
CARPETA_ET_CORREGIDA = DIR_CLIMA_TOTAL / "ET_HARGREAVES_CORREGIDA_QDM"
CARPETA_ET_CORREGIDA.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Si las temperaturas corregidas ya están en Celsius, dejar True.
# Si estuvieran en Kelvin, poner False para convertirlas.
# ----------------------------------------------------------
ENTRADA_EN_CELSIUS = True

# ----------------------------------------------------------
# Si ya existe un archivo ET:
# False -> lo salta
# True  -> lo recalcula
# ----------------------------------------------------------
SOBREESCRIBIR = False

# ----------------------------------------------------------
# Valor nodata de salida para ET
# ----------------------------------------------------------
NODATA_OUT = -9999.0

# ----------------------------------------------------------
# Cantidad de últimos archivos ET que se borrarán si detecta
# una carpeta ya iniciada al momento de reiniciar.
# ----------------------------------------------------------
N_RECALCULAR_REINICIO = 2


# ==========================================================
# 2) UTILIDADES DE CONSOLA
# ==========================================================
# Estas funciones ayudan a imprimir mensajes más claros en pantalla.

def ts() -> str:
    """Devuelve hora actual en formato HH:MM:SS."""
    return datetime.now().strftime("%H:%M:%S")


def linea(char="─", n=72):
    """Imprime una línea decorativa."""
    print(char * n)


def header(texto, char="═"):
    """Imprime un encabezado visual."""
    linea(char)
    print(f"  {texto}")
    linea(char)


def ok(texto):
    """Mensaje de operación exitosa."""
    print(f"  ✔  {texto}")


def warn(texto):
    """Mensaje de advertencia."""
    print(f"  ⚠  {texto}")


def err(texto):
    """Mensaje de error."""
    print(f"  ✘  {texto}")


def inicio_grupo(escenario, total, dir_out):
    """
    Muestra el inicio del procesamiento de un escenario/modelo.
    """
    linea()
    print(f"  ▶ INICIO   [{ts()}]")
    print(f"     Escenario : {escenario}")
    print(f"     Archivos  : {total} mes(es) en cola")
    print(f"     Salida    : {dir_out}")
    linea()


def fin_grupo(escenario, procesados, saltados, omitidos, errores, total):
    """
    Muestra el resumen final del escenario/modelo.
    """
    estado = "✔ SIN ERRORES" if errores == 0 else f"⚠ {errores} ERROR(ES)"
    linea()
    print(f"  ■ FIN      [{ts()}]  →  {estado}")
    print(f"     Escenario  : {escenario}")
    print(f"     Procesados : {procesados}")
    print(f"     Ya existían: {saltados}")
    print(f"     Omitidos   : {omitidos}")
    print(f"     Errores    : {errores}")
    print(f"     Total      : {total}")
    linea()
    print()


# ==========================================================
# 3) UTILIDADES DE RUTAS
# ==========================================================
def primera_ruta_existente(lista_rutas):
    """
    Recorre una lista de rutas y devuelve la primera que exista.

    Si ninguna existe, devuelve None.
    """
    for ruta in lista_rutas:
        if ruta.exists():
            return ruta
    return None


# Detectar automáticamente dónde está tasmax corregida
DIR_TMAX = primera_ruta_existente(CANDIDATAS_TMAX)

# Detectar automáticamente dónde está tasmin corregida
DIR_TMIN = primera_ruta_existente(CANDIDATAS_TMIN)

# Validar existencia real
if DIR_TMAX is None:
    raise FileNotFoundError(
        "No se encontró carpeta de tasmax corregida. Revisar:\n" +
        "\n".join(str(p) for p in CANDIDATAS_TMAX)
    )

if DIR_TMIN is None:
    raise FileNotFoundError(
        "No se encontró carpeta de tasmin corregida. Revisar:\n" +
        "\n".join(str(p) for p in CANDIDATAS_TMIN)
    )

ok(f"TMAX corregida detectada: {DIR_TMAX}")
ok(f"TMIN corregida detectada: {DIR_TMIN}")
ok(f"Carpeta de ET corregida : {CARPETA_ET_CORREGIDA}")
print()


# ==========================================================
# 4) UTILIDADES DE NOMBRES Y BÚSQUEDA
# ==========================================================
def existe_archivo(path: Path) -> bool:
    """
    Devuelve True si el archivo existe y NO se desea sobrescribir.
    """
    return path.exists() and (not SOBREESCRIBIR)


def extraer_anio_mes(nombre_archivo: str):
    """
    Extrae año y mes desde el nombre del archivo.

    Busca patrones del tipo:
      ..._2015_01.tif
      ..._2015_01_algo.tif
      ..._2015_01_15.tif

    Retorna:
      (anio, mes) o (None, None)
    """
    m = re.search(r'(\d{4})_(\d{2})(?:\D|$)', nombre_archivo)
    if m:
        anio = int(m.group(1))
        mes = int(m.group(2))
        if 1 <= mes <= 12:
            return anio, mes
    return None, None


def listar_tifs_por_anio_mes(carpeta: Path):
    """
    Lee todos los TIFF de una carpeta y arma un diccionario:

      {(anio, mes): Path_al_archivo}

    Esto permite emparejar fácilmente tasmax y tasmin.
    """
    salida = {}
    for p in sorted(carpeta.glob("*.tif")):
        anio, mes = extraer_anio_mes(p.name)
        if anio is not None:
            salida[(anio, mes)] = p
    return salida


def escenarios_comunes(dir_a: Path, dir_b: Path):
    """
    Busca subcarpetas comunes entre la carpeta de tasmax y tasmin.

    Ejemplo:
      EC_Earth3_ssp245
      EC_Earth3_ssp585
      MPI_ESM1_2_LR_ssp245
      ...

    Retorna una lista de pares:
      [(sub_tmax, sub_tmin), ...]
    """
    a = {p.name: p for p in dir_a.iterdir() if p.is_dir()}
    b = {p.name: p for p in dir_b.iterdir() if p.is_dir()}
    comunes = sorted(set(a) & set(b))
    return [(a[n], b[n]) for n in comunes]


# ==========================================================
# 5) REGISTRO LIVIANO EN TXT
# ==========================================================
# Este TXT guarda los nombres de los TIFF ET ya creados.
# Sirve para:
# - saber qué se llegó a procesar
# - reconstruir el estado
# - manejar el reinicio seguro

def ruta_registro_txt(dir_out: Path) -> Path:
    """Devuelve la ruta del TXT de control para una carpeta de salida."""
    return dir_out / "_registro_et_procesados.txt"


def leer_registro_txt(dir_out: Path) -> list:
    """
    Lee el TXT de registro y devuelve la lista de nombres guardados.
    Si no existe, devuelve lista vacía.
    """
    ruta_txt = ruta_registro_txt(dir_out)
    if not ruta_txt.exists():
        return []

    try:
        with open(ruta_txt, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]
    except Exception as e:
        warn(f"No se pudo leer el TXT de registro: {e}")
        return []


def reescribir_registro_txt(dir_out: Path, lista_archivos: list):
    """
    Reescribe completamente el TXT con la lista actual de archivos.

    Esto se usa especialmente después de borrar los últimos dos,
    para que el registro quede consistente con lo que realmente
    existe en disco.
    """
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "w", encoding="utf-8") as f:
            for nombre in lista_archivos:
                f.write(f"{nombre}\n")
    except Exception as e:
        warn(f"No se pudo reescribir el TXT de registro: {e}")


def append_registro_txt(dir_out: Path, nombre_archivo: str):
    """
    Agrega al final del TXT el nombre del archivo recién procesado.
    """
    ruta_txt = ruta_registro_txt(dir_out)
    try:
        with open(ruta_txt, "a", encoding="utf-8") as f:
            f.write(f"{nombre_archivo}\n")
    except Exception as e:
        warn(f"No se pudo actualizar el TXT de registro: {e}")


# ==========================================================
# 6) REINICIO SEGURO
# ==========================================================
# Lógica:
# - Si ya existen resultados en una carpeta de salida,
# - se borran SOLO los 2 últimos TIFF ET creados,
# - y luego se reconstruye el TXT.
#
# Esto permite recalcular los archivos que pudieron quedar mal
# si la ejecución anterior se interrumpió justo al final.

def listar_et_por_fecha(dir_out: Path) -> list:
    """
    Lista solo los TIFF de ET de esa carpeta, ordenados por fecha
    de modificación en disco.
    """
    archivos = [
        p for p in dir_out.glob("ET_HARGREAVES_DAILY_*.tif")
        if p.is_file()
    ]
    return sorted(archivos, key=lambda p: p.stat().st_mtime)


def preparar_reinicio_seguro(dir_out: Path):
    """
    Borra los N últimos TIFF de ET si ya existen resultados previos.

    Después reconstruye el TXT para que coincida exactamente con
    los archivos que siguen existiendo.
    """
    archivos_et = listar_et_por_fecha(dir_out)

    # Si no hay nada previo, no hay nada que borrar
    if len(archivos_et) == 0:
        return

    # Tomar solo los N últimos archivos
    ultimos = archivos_et[-N_RECALCULAR_REINICIO:]
    eliminados = []

    for ruta_arch in ultimos:
        try:
            ruta_arch.unlink()
            eliminados.append(ruta_arch.name)
        except Exception as e:
            warn(f"No se pudo borrar {ruta_arch.name}: {e}")

    if eliminados:
        warn(f"Reinicio detectado: se borraron los últimos {len(eliminados)} archivo(s) ET.")
        for nombre in eliminados:
            print(f"     - Eliminado: {nombre}")

    # Reconstruir el TXT en base a lo que realmente quedó en disco
    restantes = listar_et_por_fecha(dir_out)
    reescribir_registro_txt(dir_out, [p.name for p in restantes])


# ==========================================================
# 7) MANEJO DE FECHAS DE BANDAS
# ==========================================================
def fechas_desde_descripciones_o_mes(src, anio: int, mes: int):
    """
    Obtiene la fecha asociada a cada banda del TIFF.

    Estrategia:
    1. Si las bandas tienen descripción tipo YYYYMMDD, la usa.
    2. Si no existe esa descripción, infiere la fecha por orden:
         banda 1 -> día 1
         banda 2 -> día 2
         etc.

    Esto funciona porque los TIFF mensuales deberían tener bandas
    diarias ordenadas cronológicamente.
    """
    descs = list(src.descriptions) if src.descriptions else []

    fechas = []
    if len(descs) == src.count and all(d is not None and str(d).strip() != "" for d in descs):
        ok_parse = True
        for d in descs:
            try:
                fechas.append(datetime.strptime(str(d), "%Y%m%d"))
            except Exception:
                ok_parse = False
                break
        if ok_parse:
            return fechas

    # Si no hay descripciones válidas, inferir fechas por posición
    n_dias_mes = calendar.monthrange(anio, mes)[1]

    if src.count > n_dias_mes:
        raise ValueError(
            f"El archivo tiene {src.count} bandas, pero {anio}-{mes:02d} tiene {n_dias_mes} días."
        )

    return [datetime(anio, mes, d) for d in range(1, src.count + 1)]


# ==========================================================
# 8) CÁLCULO DE RA Y ET
# ==========================================================
def lat_rad_por_fila(transform, height: int):
    """
    Calcula la latitud en radianes para cada fila del raster.

    ¿Por qué por fila?
    Porque la radiación extraterrestre (Ra) depende de la latitud,
    y la latitud cambia verticalmente en la imagen.

    Devuelve un array columna (height x 1).
    """
    filas = np.arange(height, dtype=np.float64)

    # transform.f -> coordenada Y superior
    # transform.e -> tamaño de pixel en Y (normalmente negativo)
    # filas + 0.5 -> centro de pixel
    lat_deg = transform.f + transform.e * (filas + 0.5)

    return np.deg2rad(lat_deg).reshape(-1, 1)


def ra_hargreaves_por_fila(lat_rad_2d, doy: int):
    """
    Calcula la radiación extraterrestre (Ra) para un día del año.

    Usa la misma lógica que tu script original de Earth Engine:
      delta = 0.409 * sin(2*pi*doy/365 - 1.39)
      omega_s = acos(clamp(-tan(phi)*tan(delta), -1, 1))
      ra = 37.6 * (...)

    Parámetros:
    -----------
    lat_rad_2d : array
        Latitud por fila en radianes.
    doy : int
        Día del año.

    Retorna:
    --------
    np.ndarray con Ra por fila.
    """
    delta = 0.409 * np.sin((2.0 * math.pi * doy / 365.0) - 1.39)

    cos_omega_s = -np.tan(lat_rad_2d) * math.tan(delta)
    cos_omega_s = np.clip(cos_omega_s, -1.0, 1.0)
    omega_s = np.arccos(cos_omega_s)

    ra = 37.6 * (
        omega_s * np.sin(lat_rad_2d) * math.sin(delta)
        + np.cos(lat_rad_2d) * math.cos(delta) * np.sin(omega_s)
    )

    return ra.astype(np.float32)


def calcular_et_hargreaves_numpy(tmax, tmin, ra_2d, nodata_mask):
    """
    Calcula ET diaria usando Hargreaves-Samani.

    Fórmula aplicada:
      t_mean  = (tmax + tmin) / 2
      t_range = |tmax - tmin|
      ET      = Ra * 0.0009384 * (t_mean + 17.8) * sqrt(t_range)

    Solo escribe resultados en píxeles válidos.
    Los píxeles inválidos quedan con NODATA_OUT.
    """
    t_mean = (tmax + tmin) / 2.0
    t_range = np.abs(tmax - tmin)

    et = ra_2d * 0.0009384 * (t_mean + 17.8) * np.sqrt(t_range)
    et = et.astype(np.float32)

    salida = np.full(tmax.shape, NODATA_OUT, dtype=np.float32)
    salida[~nodata_mask] = et[~nodata_mask]

    return salida


# ==========================================================
# 9) PROCESAMIENTO DE UN ESCENARIO/MODELO
# ==========================================================
def procesar_escenario(dir_tmax_esc: Path, dir_tmin_esc: Path, dir_out_esc: Path):
    """
    Procesa un escenario/modelo completo.

    Ejemplo de escenario:
      EC_Earth3_ssp245
      EC_Earth3_ssp585
      MPI_ESM1_2_LR_ssp245
      etc.

    Pasos internos:
    ---------------
    1. Crear carpeta de salida.
    2. Buscar meses comunes entre tasmax y tasmin.
    3. Aplicar reinicio seguro.
    4. Mostrar barra de progreso.
    5. Para cada mes:
         - abrir tasmax/tasmin
         - validar consistencia
         - calcular ET banda por banda
         - guardar TIFF de salida
         - actualizar TXT
    """
    dir_out_esc.mkdir(parents=True, exist_ok=True)

    # Listar TIFF mensuales disponibles para tasmax y tasmin
    tmax_files = listar_tifs_por_anio_mes(dir_tmax_esc)
    tmin_files = listar_tifs_por_anio_mes(dir_tmin_esc)

    # Solo se procesan meses presentes en ambas variables
    claves = sorted(set(tmax_files.keys()) & set(tmin_files.keys()))

    if not claves:
        warn(f"Sin pares mensuales comunes en: {dir_tmax_esc.name}")
        return

    # Aplicar reinicio seguro antes de comenzar el escenario
    preparar_reinicio_seguro(dir_out_esc)

    # Volver a leer el TXT por si ya existían archivos procesados
    ya_registrados = set(leer_registro_txt(dir_out_esc))

    total = len(claves)
    procesados = 0
    saltados = 0
    omitidos = 0
    errores = 0

    inicio_grupo(dir_tmax_esc.name, total, dir_out_esc)

    barra = tqdm(
        claves,
        total=total,
        desc=f"  ET | {dir_tmax_esc.name}",
        unit="mes",
        colour="green"
    )

    for (anio, mes) in barra:
        nombre_out = f"ET_HARGREAVES_DAILY_{dir_tmax_esc.name}_{anio}_{mes:02d}_WGS84_GRID90m.tif"
        out_path = dir_out_esc / nombre_out

        # Si el archivo ya existe y no se va a sobrescribir, se salta
        if existe_archivo(out_path):
            saltados += 1
            barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)
            continue

        # Rutas de entrada
        ruta_tmax = tmax_files.get((anio, mes))
        ruta_tmin = tmin_files.get((anio, mes))

        # Si por alguna razón falta uno de los dos, omitir
        if ruta_tmax is None or ruta_tmin is None:
            omitidos += 1
            barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)
            continue

        try:
            with rasterio.open(ruta_tmax) as src_max, rasterio.open(ruta_tmin) as src_min:

                # --------------------------------------------------
                # Validaciones de consistencia entre tasmax y tasmin
                # --------------------------------------------------
                if src_max.count != src_min.count:
                    raise ValueError(
                        f"Número de bandas distinto: tmax={src_max.count}, tmin={src_min.count}"
                    )

                if src_max.width != src_min.width or src_max.height != src_min.height:
                    raise ValueError("Dimensiones distintas entre tasmax y tasmin.")

                if src_max.transform != src_min.transform:
                    raise ValueError("Transform distinta entre tasmax y tasmin.")

                # --------------------------------------------------
                # Preparar metadatos de salida
                # --------------------------------------------------
                meta = src_max.meta.copy()
                meta.update(
                    dtype=rasterio.float32,
                    count=src_max.count,
                    nodata=NODATA_OUT,
                    compress="lzw"
                )

                # Recuperar fechas de cada banda
                fechas = fechas_desde_descripciones_o_mes(src_max, anio, mes)

                # Calcular latitud por fila una sola vez por archivo
                lat_rad_2d = lat_rad_por_fila(src_max.transform, src_max.height)

                # Guardar nodata originales
                nodata_max = src_max.nodata
                nodata_min = src_min.nodata

                # --------------------------------------------------
                # Crear archivo ET de salida
                # --------------------------------------------------
                with rasterio.open(out_path, "w", **meta) as dst:
                    for b in range(1, src_max.count + 1):
                        fecha = fechas[b - 1]
                        doy = fecha.timetuple().tm_yday

                        # Radiación extraterrestre para ese día
                        ra_2d = ra_hargreaves_por_fila(lat_rad_2d, doy)

                        # Leer banda diaria de tasmax y tasmin
                        tmax = src_max.read(b).astype(np.float32)
                        tmin = src_min.read(b).astype(np.float32)

                        # Convertir a Celsius si hiciera falta
                        if not ENTRADA_EN_CELSIUS:
                            tmax = tmax - 273.15
                            tmin = tmin - 273.15

                        # Máscara de nodata e inválidos
                        nodata_mask = ~np.isfinite(tmax) | ~np.isfinite(tmin)

                        if nodata_max is not None:
                            nodata_mask |= (tmax == nodata_max)

                        if nodata_min is not None:
                            nodata_mask |= (tmin == nodata_min)

                        # Calcular ET diaria
                        et = calcular_et_hargreaves_numpy(
                            tmax=tmax,
                            tmin=tmin,
                            ra_2d=ra_2d,
                            nodata_mask=nodata_mask
                        )

                        # Escribir banda en salida
                        dst.write(et, b)

                        # Guardar fecha como descripción de banda
                        dst.set_band_description(b, fecha.strftime("%Y%m%d"))

                # --------------------------------------------------
                # Si llegó aquí, el archivo mensual se procesó bien
                # --------------------------------------------------
                procesados += 1

                if nombre_out not in ya_registrados:
                    append_registro_txt(dir_out_esc, nombre_out)
                    ya_registrados.add(nombre_out)

        except Exception as e:
            errores += 1
            err(f"[ERROR {errores}] {dir_tmax_esc.name} | {anio}-{mes:02d}: {e}")

        barra.set_postfix(proc=procesados, skip=saltados, omi=omitidos, err=errores)

    barra.close()

    # Reconstrucción final del TXT desde el disco real
    restantes = listar_et_por_fecha(dir_out_esc)
    reescribir_registro_txt(dir_out_esc, [p.name for p in restantes])

    fin_grupo(dir_tmax_esc.name, procesados, saltados, omitidos, errores, total)


# ==========================================================
# 10) EJECUCIÓN PRINCIPAL
# ==========================================================
header("ET HARGREAVES DESDE TEMPERATURA CORREGIDA", char="█")

# Buscar carpetas comunes entre tasmax corregida y tasmin corregida
pares_escenarios = escenarios_comunes(DIR_TMAX, DIR_TMIN)

if not pares_escenarios:
    raise RuntimeError("No se encontraron carpetas comunes de escenario entre tasmax y tasmin.")

# Procesar cada escenario/modelo
for dir_tmax_esc, dir_tmin_esc in pares_escenarios:
    nombre_escenario = dir_tmax_esc.name
    dir_out_esc = CARPETA_ET_CORREGIDA / nombre_escenario

    print(f"\nProcesando escenario/modelo: {nombre_escenario}")
    print(f"  TMAX: {dir_tmax_esc}")
    print(f"  TMIN: {dir_tmin_esc}")
    print(f"  OUT : {dir_out_esc}")

    procesar_escenario(dir_tmax_esc, dir_tmin_esc, dir_out_esc)

header("✔ PROCESO COMPLETO FINALIZADO EXITOSAMENTE", char="█")
print(f"  TMAX corregida : {DIR_TMAX}")
print(f"  TMIN corregida : {DIR_TMIN}")
print(f"  ET corregida   : {CARPETA_ET_CORREGIDA}")